# KG1 V207B External Adapter Triage Colab

Purpose: continue the V207 roadmap after V206B/V206C/V214 were rejected.

This notebook:

- reuses V207A official-like validation artifacts when they exist in Drive;
- bootstraps the validated V194 baseline exports when those artifacts are missing;
- downloads public Kaggle model adapters into Drive for gated testing;
- audits external/current adapter structures before spending H100/A100 time;
- screens only the weak families first: `equation_transform` and `bit_manipulation`;
- runs full 947-row official-like ACC only for weak-positive candidates;
- never trains and never submits to Kaggle.

Logging policy: cell output is intentionally concise. The notebook prints only
start/end markers, key Drive paths, command return codes, elapsed time,
artifact counts, and eval/gate summaries. Full subprocess logs are stored under
`/content/drive/MyDrive/KG1_NVIDIA_V207B/output_v207b_external_adapter_triage/reports`.

Drive layout: persistent run outputs stay under `KG1_NVIDIA_V207B`, public
adapter downloads stay under `KG1_PUBLIC_ADAPTERS`, and compact manifests stay
under `KG1_NVIDIA_V207B/output_v207b_external_adapter_triage/manifests`.

Colab URL:

`https://colab.research.google.com/github/FELIPEACASTRO/KG1-NVIDIA/blob/v207b-external-triage/notebooks/KG1_V207B_EXTERNAL_ADAPTER_TRIAGE_COLAB.ipynb`


In [1]:
# CELL: mount Drive.
print('=== V207B DRIVE MOUNT START ===')
from google.colab import drive
drive.mount('/content/drive')
print('=== V207B DRIVE MOUNT END ===')


=== V207B DRIVE MOUNT START ===
Mounted at /content/drive
=== V207B DRIVE MOUNT END ===


In [2]:
# CELL: runtime configuration.
print('=== V207B CONFIG START ===')
import datetime
import hashlib
import json
import os
import pathlib
import re
import shutil
import subprocess
import sys
import time

VERSION = 'V207B_EXTERNAL_ADAPTER_TRIAGE_20260507_PUBLIC_DOWNLOADS_R2'
ROOT = pathlib.Path('/content/kg1')
DRIVE_MY = pathlib.Path('/content/drive/MyDrive')
V207A_ROOT = DRIVE_MY / 'KG1_NVIDIA_V207A' / 'output_v207a_acc_gate'
OUT_ROOT = DRIVE_MY / 'KG1_NVIDIA_V207B' / 'output_v207b_external_adapter_triage'
REPORT_DIR = OUT_ROOT / 'reports'
PUBLIC_KAGGLE_ROOT = DRIVE_MY / 'KG1_PUBLIC_ADAPTERS'
MANIFEST_DIR = OUT_ROOT / 'manifests'
FALLBACK_EXPORT_BASE = os.environ.get(
    'KG1_V207B_FALLBACK_EXPORT_BASE',
    'https://raw.githubusercontent.com/FELIPEACASTRO/KG1-NVIDIA/v207b-external-triage/artifacts/drive_exports',
)
MODEL_NAME = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'
MODEL_REVISION = 'cbd3fa9f933d55ef16a84236559f4ee2a0526848'
VLLM_VERSION = '0.20.1'
VLLM_CUDA_FLAVOR = 'cu129'
PYTORCH_CUDA_INDEX_URL = os.environ.get(
    'KG1_V207B_PYTORCH_CUDA_INDEX_URL',
    'https://download.pytorch.org/whl/cu129',
)
VLLM_WHEEL_URL = os.environ.get(
    'KG1_V207B_VLLM_WHEEL_URL',
    'https://github.com/vllm-project/vllm/releases/download/v0.20.1/vllm-0.20.1%2Bcu129-cp38-abi3-manylinux_2_31_x86_64.whl',
)
VLLM_SPEC = os.environ.get('KG1_V207B_VLLM_SPEC', VLLM_WHEEL_URL)
FORCE_VLLM_REINSTALL = os.environ.get('KG1_V207B_FORCE_VLLM_REINSTALL', '1') == '1'
VLLM_INSTALL_POLICY = (
    'vllm==0.20.1 via official cu129 wheel; uninstall stale vllm first; '
    'subprocess import preflight must pass before any adapter evaluation'
)

VAL_CSV = V207A_ROOT / 'validation' / 'official_train_seed42_stratified10_val.csv'
VAL_WEAK_CSV = OUT_ROOT / 'validation' / 'official_train_seed42_stratified10_val_weak_families.csv'
BASELINE_PREDICTIONS = V207A_ROOT / 'v194_baseline_eval' / 'v194_baseline_predictions.csv'
BASELINE_PER_TASK = V207A_ROOT / 'v194_baseline_eval' / 'v194_baseline_per_task.csv'
BASELINE_REPORT = V207A_ROOT / 'v194_baseline_eval' / 'v194_baseline_eval_report.json'

WEAK_FAMILIES = ['bit_manipulation', 'equation_transform']
FORCE_REEVAL = os.environ.get('KG1_V207B_FORCE_REEVAL', '0') == '1'
RUN_FULL_FOR_POSITIVE = os.environ.get('KG1_V207B_RUN_FULL_FOR_POSITIVE', '1') == '1'
HASH_WEIGHTS = os.environ.get('KG1_V207B_HASH_WEIGHTS', '0') == '1'
MAX_DISCOVERY_DIRS = int(os.environ.get('KG1_V207B_MAX_DISCOVERY_DIRS', '25000'))
INCLUDE_REJECTED_V206 = os.environ.get('KG1_V207B_INCLUDE_REJECTED_V206', '0') == '1'
RUN_KAGGLE_PUBLIC_DOWNLOAD = os.environ.get('KG1_V207B_RUN_KAGGLE_PUBLIC_DOWNLOAD', '1') == '1'
PUBLIC_DOWNLOAD_MAX_PRIORITY = int(os.environ.get('KG1_V207B_PUBLIC_DOWNLOAD_MAX_PRIORITY', '2'))
PUBLIC_DOWNLOAD_MAX_CANDIDATES = int(os.environ.get('KG1_V207B_PUBLIC_DOWNLOAD_MAX_CANDIDATES', '13'))
LOG_MODE = os.environ.get('KG1_V207B_LOG_MODE', 'essential').strip().lower()
PRINT_COMMAND_OUTPUT = LOG_MODE in ('verbose', 'debug')
LOG_TAIL_LINES = int(os.environ.get('KG1_V207B_LOG_TAIL_LINES', '30'))
LOG_POLICY = (
    'essential: cell START/END, key Drive paths, command line/rc/elapsed, '
    'artifact counts, eval/gate summaries; full subprocess stdout goes to Drive log files'
)
ALLOW_KAGGLE_SUBMIT = False

for path in [OUT_ROOT, REPORT_DIR, MANIFEST_DIR, VAL_WEAK_CSV.parent, PUBLIC_KAGGLE_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print('config =', json.dumps({
    'VERSION': VERSION,
    'ROOT': str(ROOT),
    'OUT_ROOT': str(OUT_ROOT),
    'REPORT_DIR': str(REPORT_DIR),
    'MANIFEST_DIR': str(MANIFEST_DIR),
    'PUBLIC_KAGGLE_ROOT': str(PUBLIC_KAGGLE_ROOT),
    'VAL_CSV': str(VAL_CSV),
    'BASELINE_PREDICTIONS': str(BASELINE_PREDICTIONS),
    'MODEL_NAME': MODEL_NAME,
    'MODEL_REVISION': MODEL_REVISION,
    'VLLM_VERSION': VLLM_VERSION,
    'VLLM_CUDA_FLAVOR': VLLM_CUDA_FLAVOR,
    'PYTORCH_CUDA_INDEX_URL': PYTORCH_CUDA_INDEX_URL,
    'VLLM_WHEEL_URL': VLLM_WHEEL_URL,
    'VLLM_SPEC': VLLM_SPEC,
    'FORCE_VLLM_REINSTALL': FORCE_VLLM_REINSTALL,
    'VLLM_INSTALL_POLICY': VLLM_INSTALL_POLICY,
    'WEAK_FAMILIES': WEAK_FAMILIES,
    'RUN_KAGGLE_PUBLIC_DOWNLOAD': RUN_KAGGLE_PUBLIC_DOWNLOAD,
    'PUBLIC_DOWNLOAD_MAX_PRIORITY': PUBLIC_DOWNLOAD_MAX_PRIORITY,
    'PUBLIC_DOWNLOAD_MAX_CANDIDATES': PUBLIC_DOWNLOAD_MAX_CANDIDATES,
    'RUN_FULL_FOR_POSITIVE': RUN_FULL_FOR_POSITIVE,
    'FORCE_REEVAL': FORCE_REEVAL,
    'HASH_WEIGHTS': HASH_WEIGHTS,
    'LOG_MODE': LOG_MODE,
    'PRINT_COMMAND_OUTPUT': PRINT_COMMAND_OUTPUT,
    'ALLOW_KAGGLE_SUBMIT': ALLOW_KAGGLE_SUBMIT,
}, indent=2, sort_keys=True))
print('LOG_POLICY =', LOG_POLICY)
print('VLLM_INSTALL_POLICY =', VLLM_INSTALL_POLICY)
print('DRIVE_ORGANIZED_OUTPUTS =', json.dumps({
    'run_outputs': str(OUT_ROOT),
    'logs': str(REPORT_DIR),
    'manifests': str(MANIFEST_DIR),
    'downloaded_public_adapters': str(PUBLIC_KAGGLE_ROOT),
}, indent=2, sort_keys=True))
if ALLOW_KAGGLE_SUBMIT:
    raise RuntimeError('This notebook is submit-disabled by design.')
print('=== V207B CONFIG END ===')


=== V207B CONFIG START ===
config = {
  "ALLOW_KAGGLE_SUBMIT": false,
  "BASELINE_PREDICTIONS": "/content/drive/MyDrive/KG1_NVIDIA_V207A/output_v207a_acc_gate/v194_baseline_eval/v194_baseline_predictions.csv",
  "FORCE_REEVAL": false,
  "FORCE_VLLM_REINSTALL": true,
  "HASH_WEIGHTS": false,
  "LOG_MODE": "essential",
  "MANIFEST_DIR": "/content/drive/MyDrive/KG1_NVIDIA_V207B/output_v207b_external_adapter_triage/manifests",
  "MODEL_NAME": "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16",
  "MODEL_REVISION": "cbd3fa9f933d55ef16a84236559f4ee2a0526848",
  "OUT_ROOT": "/content/drive/MyDrive/KG1_NVIDIA_V207B/output_v207b_external_adapter_triage",
  "PRINT_COMMAND_OUTPUT": false,
  "PUBLIC_DOWNLOAD_MAX_CANDIDATES": 13,
  "PUBLIC_DOWNLOAD_MAX_PRIORITY": 2,
  "PUBLIC_KAGGLE_ROOT": "/content/drive/MyDrive/KG1_PUBLIC_ADAPTERS",
  "PYTORCH_CUDA_INDEX_URL": "https://download.pytorch.org/whl/cu129",
  "REPORT_DIR": "/content/drive/MyDrive/KG1_NVIDIA_V207B/output_v207b_external_adapter_triage/reports",

In [3]:
# CELL: bridge Colab Secrets into environment variables without printing secret values.
print('=== V207B COLAB SECRETS BRIDGE START ===')
import json
import os
import pathlib

def read_colab_secret(name):
    try:
        from google.colab import userdata  # type: ignore
    except Exception:
        return None
    try:
        value = userdata.get(name)
    except Exception:
        return None
    if value is None:
        return None
    value = str(value).strip()
    return value or None

def set_env_from_secret(env_name, secret_names):
    if os.environ.get(env_name):
        print(env_name, 'already_set=True')
        return True
    for secret_name in secret_names:
        value = read_colab_secret(secret_name)
        print('colab_secret_available', secret_name, '=', bool(value))
        if value:
            os.environ[env_name] = value
            print(env_name, 'set_from_secret=', secret_name)
            return True
    print(env_name, 'set_from_secret=False')
    return False

set_env_from_secret('HF_TOKEN', ['HF_TOKEN', 'HF_KEY'])
if os.environ.get('HF_TOKEN'):
    os.environ.setdefault('HUGGINGFACE_HUB_TOKEN', os.environ['HF_TOKEN'])
    os.environ.setdefault('HF_KEY', os.environ['HF_TOKEN'])
    print('HF_TOKEN_ready=True')
else:
    print('HF_TOKEN_ready=False')

set_env_from_secret('KAGGLE_USERNAME', ['KAGGLE_USERNAME'])
set_env_from_secret('KAGGLE_KEY', ['KAGGLE_KEY'])
kaggle_dir = pathlib.Path('/root/.kaggle')
kaggle_json = kaggle_dir / 'kaggle.json'
if os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'):
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    if not kaggle_json.exists():
        kaggle_json.write_text(
            json.dumps(
                {
                    'username': os.environ['KAGGLE_USERNAME'],
                    'key': os.environ['KAGGLE_KEY'],
                }
            ),
            encoding='utf-8',
        )
        print('kaggle_json_created_from_colab_secrets=True')
    else:
        print('kaggle_json_already_exists=True')
    kaggle_json.chmod(0o600)
    os.environ.setdefault('KAGGLE_CONFIG_DIR', str(kaggle_dir))
    print('KAGGLE_CONFIG_DIR =', os.environ.get('KAGGLE_CONFIG_DIR'))
    print('KAGGLE_CREDENTIALS_READY=True')
else:
    print('KAGGLE_CREDENTIALS_READY=False')
print('=== V207B COLAB SECRETS BRIDGE END ===')


=== V207B COLAB SECRETS BRIDGE START ===
colab_secret_available HF_TOKEN = True
HF_TOKEN set_from_secret= HF_TOKEN
HF_TOKEN_ready=True
colab_secret_available KAGGLE_USERNAME = True
KAGGLE_USERNAME set_from_secret= KAGGLE_USERNAME
colab_secret_available KAGGLE_KEY = True
KAGGLE_KEY set_from_secret= KAGGLE_KEY
kaggle_json_created_from_colab_secrets=True
KAGGLE_CONFIG_DIR = /root/.kaggle
KAGGLE_CREDENTIALS_READY=True
=== V207B COLAB SECRETS BRIDGE END ===


In [4]:
# CELL: helper functions and command logging.
print('=== V207B HELPERS START ===')
import importlib
import json
import os
import pathlib
import subprocess
import sys
import time

ESSENTIAL_OUTPUT_PATTERNS = [
    'Traceback',
    'RuntimeError',
    'Error:',
    'ERROR',
    'failed',
    'returncode =',
    'vLLM loaded',
    'warmup_elapsed_s',
    'generation_elapsed_s',
    'raw_predictions_pre_score_csv',
    'summary =',
    'accuracy',
    'correct',
    'truncated',
    'predictions_csv',
    'per_task_csv',
    'report_json',
    'tokens_per_second',
    'heartbeat',
    'fresh_python_import_ok',
    'vllm_import_preflight',
    'VLLM_INSTALL_POLICY',
    'libcudart.so',
]

def is_essential_output_line(line):
    return any(pattern in line for pattern in ESSENTIAL_OUTPUT_PATTERNS)

def run_cmd(cmd, cwd=None, env=None, log_path=None, check=True):
    cmd = [str(x) for x in cmd]
    started = time.time()
    print('+', ' '.join(cmd))
    if cwd:
        print('cwd =', cwd)
    log_handle = None
    if log_path is not None:
        log_path = pathlib.Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_handle = log_path.open('w', encoding='utf-8')
        print('log_path =', log_path)
    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    suppressed_lines = 0
    tail_lines = []
    for line in proc.stdout:
        tail_lines.append(line.rstrip('\n'))
        if len(tail_lines) > LOG_TAIL_LINES:
            tail_lines.pop(0)
        if PRINT_COMMAND_OUTPUT or is_essential_output_line(line):
            print(line, end='')
        else:
            suppressed_lines += 1
        if log_handle:
            log_handle.write(line)
    rc = proc.wait()
    if log_handle:
        log_handle.close()
    elapsed = time.time() - started
    print('returncode =', rc)
    print('elapsed_s =', round(elapsed, 1))
    print('command_output_suppressed_lines =', suppressed_lines)
    if log_path is not None:
        print('full_command_log =', log_path)
    if rc != 0 and tail_lines:
        print('command_tail_on_failure =')
        print('\n'.join(tail_lines[-LOG_TAIL_LINES:]))
    if check and rc != 0:
        raise RuntimeError(f'Command failed with rc={rc}: {cmd}')
    return rc

def ensure_import(import_name, pip_spec=None):
    try:
        mod = importlib.import_module(import_name)
        print(import_name, 'version=', getattr(mod, '__version__', 'unknown'))
        return mod
    except Exception as exc:
        print(import_name, 'missing/import failed:', repr(exc))
        if not pip_spec:
            raise
        run_cmd([sys.executable, '-m', 'pip', 'install', '-q', pip_spec])
        mod = importlib.import_module(import_name)
        print(import_name, 'version=', getattr(mod, '__version__', 'unknown'))
        return mod

def fresh_python_import_check(import_name, log_path=None, check=True):
    code = (
        "import importlib, torch; "
        f"m=importlib.import_module('{import_name}'); "
        "print('fresh_python_import_ok', m.__name__, getattr(m, '__version__', 'unknown')); "
        "print('fresh_python_torch', torch.__version__, getattr(torch.version, 'cuda', 'unknown'))"
    )
    return run_cmd([sys.executable, '-c', code], log_path=log_path, check=check)

def refresh_nvidia_library_path():
    try:
        import site
        candidates = []
        for base in list(site.getsitepackages()) + [site.getusersitepackages()]:
            root = pathlib.Path(base) / 'nvidia'
            if not root.exists():
                continue
            for lib_dir in root.glob('*/lib'):
                if lib_dir.exists():
                    candidates.append(str(lib_dir))
        existing = [item for item in os.environ.get('LD_LIBRARY_PATH', '').split(':') if item]
        merged = []
        for item in candidates + existing:
            if item and item not in merged:
                merged.append(item)
        if merged:
            os.environ['LD_LIBRARY_PATH'] = ':'.join(merged)
        print('nvidia_library_path_dirs =', len(candidates))
    except Exception as exc:
        print('nvidia_library_path_refresh_failed =', repr(exc))

def install_verified_vllm():
    print('VLLM_INSTALL_POLICY =', VLLM_INSTALL_POLICY)
    print('VLLM_SPEC =', VLLM_SPEC)
    print('VLLM_WHEEL_URL =', VLLM_WHEEL_URL)
    print('PYTORCH_CUDA_INDEX_URL =', PYTORCH_CUDA_INDEX_URL)
    if FORCE_VLLM_REINSTALL:
        run_cmd(
            [sys.executable, '-m', 'pip', 'uninstall', '-y', 'vllm'],
            log_path=REPORT_DIR / 'vllm_uninstall.log',
            check=False,
        )
    run_cmd(
        [
            sys.executable,
            '-m',
            'pip',
            'install',
            '-q',
            '--no-cache-dir',
            '--extra-index-url',
            PYTORCH_CUDA_INDEX_URL,
            VLLM_SPEC,
        ],
        log_path=REPORT_DIR / 'vllm_install.log',
    )
    refresh_nvidia_library_path()
    preflight_log = REPORT_DIR / 'vllm_import_preflight.log'
    print('vllm_import_preflight_log =', preflight_log)
    rc = fresh_python_import_check('vllm', log_path=preflight_log, check=False)
    if rc != 0:
        text = preflight_log.read_text(encoding='utf-8', errors='replace') if preflight_log.exists() else ''
        if 'libcudart.so.13' in text:
            raise RuntimeError(
                'vLLM import preflight failed: CUDA 13 runtime was requested but libcudart.so.13 '
                'is not available. Restart the Colab runtime and rerun from the dependency cell; '
                'keep VLLM_SPEC on the official cu129 wheel. log=' + str(preflight_log)
            )
        raise RuntimeError('vLLM import preflight failed. log=' + str(preflight_log))
    print('vllm_import_preflight_ok = True')

def sha256_file(path, enabled=True):
    path = pathlib.Path(path)
    if not enabled:
        return ''
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def safe_label(text):
    text = str(text).strip().replace('\\', '/')
    text = re.sub(r'[^A-Za-z0-9_.-]+', '_', text)
    text = re.sub(r'_+', '_', text).strip('_.-')
    return text[-120:] or 'candidate'

print('python =', sys.version)
print('=== V207B HELPERS END ===')


=== V207B HELPERS START ===
python = 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
=== V207B HELPERS END ===


In [ ]:
# CELL: install dependencies and validate GPU/vLLM import path.
print('=== V207B DEPENDENCY CHECK START ===')
ensure_import('pandas', 'pandas')
ensure_import('huggingface_hub', 'huggingface_hub')
ensure_import('transformers', 'transformers')
ensure_import('peft', 'peft')
ensure_import('safetensors', 'safetensors')
run_cmd([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle==2.0.2'])
KAGGLE_EXE = shutil.which('kaggle')
KAGGLE_CMD_PREFIX = [KAGGLE_EXE] if KAGGLE_EXE else [sys.executable, '-m', 'kaggle.cli']
print('KAGGLE_EXE =', KAGGLE_EXE)
print('KAGGLE_CMD_PREFIX =', KAGGLE_CMD_PREFIX)
run_cmd(KAGGLE_CMD_PREFIX + ['--version'])
torch = ensure_import('torch')
print('torch_cuda_available =', torch.cuda.is_available())
print('torch_cuda_device_count =', torch.cuda.device_count() if torch.cuda.is_available() else 0)
if torch.cuda.is_available():
    print('torch_cuda_device_name =', torch.cuda.get_device_name(0))
    print('torch_cuda_version =', getattr(torch.version, 'cuda', 'unknown'))
print('Installing and verifying pinned vLLM runtime for subprocess evaluation.')
install_verified_vllm()
print('=== V207B DEPENDENCY CHECK END ===')


=== V207B DEPENDENCY CHECK START ===
pandas version= 2.2.2
huggingface_hub version= 1.11.0
transformers version= 5.0.0
peft version= 0.19.1
safetensors version= 0.7.0
+ /usr/bin/python3 -m pip install -q kaggle==2.0.2
returncode = 0
elapsed_s = 1.8
command_output_suppressed_lines = 0
KAGGLE_EXE = /usr/local/bin/kaggle
KAGGLE_CMD_PREFIX = ['/usr/local/bin/kaggle']
+ /usr/local/bin/kaggle --version
returncode = 0
elapsed_s = 0.6
command_output_suppressed_lines = 1
torch version= 2.10.0+cu128
torch_cuda_available = True
torch_cuda_device_count = 1
torch_cuda_device_name = NVIDIA H100 80GB HBM3
torch_cuda_version = 12.8
Installing and verifying pinned vLLM runtime for subprocess evaluation.
VLLM_INSTALL_POLICY = vllm==0.20.1 via official cu129 wheel; uninstall stale vllm first; subprocess import preflight must pass before any adapter evaluation
VLLM_SPEC = https://github.com/vllm-project/vllm/releases/download/v0.20.1/vllm-0.20.1%2Bcu129-cp38-abi3-manylinux_2_31_x86_64.whl
VLLM_WHEEL_URL =

In [ ]:
# CELL: create local execution workspace for embedded scripts.
print('=== V207B WORKSPACE SETUP START ===')
ROOT.mkdir(parents=True, exist_ok=True)
(ROOT / 'scripts').mkdir(parents=True, exist_ok=True)
(ROOT / 'src').mkdir(parents=True, exist_ok=True)
print('ROOT =', ROOT, 'exists=', ROOT.exists())
print('scripts_dir =', ROOT / 'scripts', 'exists=', (ROOT / 'scripts').exists())
print('src_dir =', ROOT / 'src', 'exists=', (ROOT / 'src').exists())
print('=== V207B WORKSPACE SETUP END ===')


In [ ]:
# CELL: install/repair V207B metric scripts inside the local workspace.
print('=== V207B SCRIPT BOOTSTRAP START ===')
import json, pathlib, py_compile
ROOT = pathlib.Path('/content/kg1')
FILES = json.loads("{\n  \"src/__init__.py\": \"\\\"\\\"\\\"KG1 shared Python utilities.\\\"\\\"\\\"\\n\\n\",\n  \"src/competition_utils.py\": \"\\\"\\\"\\\"Shared metric utilities for the NVIDIA Nemotron reasoning challenge.\\n\\nThe answer extraction and verification functions intentionally mirror the\\npublic Kaggle metric path used by the Jiazhuang/Xduan local-CV notebooks:\\nextract the last boxed answer first, then fall back to final-answer phrases,\\nthen the last number, then the last non-empty line.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport math\\nimport re\\nimport unicodedata\\nfrom pathlib import Path\\nfrom typing import Any\\n\\n\\nREPO_ROOT = Path(__file__).resolve().parent.parent\\nDEFAULT_DATA_DIR = REPO_ROOT / \\\"data\\\"\\n\\nMODEL_NAME = \\\"nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16\\\"\\nMODEL_REVISION = \\\"cbd3fa9f933d55ef16a84236559f4ee2a0526848\\\"\\n\\nOFFICIAL_INFERENCE_CONFIG: dict[str, Any] = {\\n    \\\"model_name\\\": MODEL_NAME,\\n    \\\"model_revision\\\": MODEL_REVISION,\\n    \\\"max_lora_rank\\\": 32,\\n    \\\"max_tokens\\\": 7680,\\n    \\\"temperature\\\": 0.0,\\n    \\\"top_p\\\": 1.0,\\n    \\\"max_model_len\\\": 8192,\\n    \\\"max_num_seqs\\\": 64,\\n    \\\"gpu_memory_utilization\\\": 0.85,\\n    \\\"enable_prefix_caching\\\": True,\\n    \\\"enable_chunked_prefill\\\": True,\\n    \\\"trust_remote_code\\\": True,\\n    \\\"dtype\\\": \\\"auto\\\",\\n}\\n\\nPROMPT_SUFFIX = (\\n    \\\"\\\\nPlease put your final answer inside `\\\\\\\\boxed{}`. \\\"\\n    \\\"For example: `\\\\\\\\boxed{your answer}`\\\"\\n)\\n\\nFAMILIES = (\\n    \\\"gravity_constant\\\",\\n    \\\"unit_conversion\\\",\\n    \\\"numeral_system\\\",\\n    \\\"text_encryption\\\",\\n    \\\"bit_manipulation\\\",\\n    \\\"equation_transform\\\",\\n)\\n\\nFAMILY_ALIASES = {\\n    \\\"gravity\\\": \\\"gravity_constant\\\",\\n    \\\"grav\\\": \\\"gravity_constant\\\",\\n    \\\"gravity_constant\\\": \\\"gravity_constant\\\",\\n    \\\"unit\\\": \\\"unit_conversion\\\",\\n    \\\"units\\\": \\\"unit_conversion\\\",\\n    \\\"unit_conversion\\\": \\\"unit_conversion\\\",\\n    \\\"numeral\\\": \\\"numeral_system\\\",\\n    \\\"roman\\\": \\\"numeral_system\\\",\\n    \\\"roman_numeral\\\": \\\"numeral_system\\\",\\n    \\\"number_system\\\": \\\"numeral_system\\\",\\n    \\\"numeral_system\\\": \\\"numeral_system\\\",\\n    \\\"cipher\\\": \\\"text_encryption\\\",\\n    \\\"encryption\\\": \\\"text_encryption\\\",\\n    \\\"text\\\": \\\"text_encryption\\\",\\n    \\\"text_cipher\\\": \\\"text_encryption\\\",\\n    \\\"text_encryption\\\": \\\"text_encryption\\\",\\n    \\\"bit\\\": \\\"bit_manipulation\\\",\\n    \\\"bits\\\": \\\"bit_manipulation\\\",\\n    \\\"bit_manipulation\\\": \\\"bit_manipulation\\\",\\n    \\\"eq\\\": \\\"equation_transform\\\",\\n    \\\"equation\\\": \\\"equation_transform\\\",\\n    \\\"equation_rules\\\": \\\"equation_transform\\\",\\n    \\\"symbol_transform\\\": \\\"equation_transform\\\",\\n    \\\"equation_symbolic\\\": \\\"equation_transform\\\",\\n    \\\"equation_numeric\\\": \\\"equation_transform\\\",\\n    \\\"equation_numeric_deduce\\\": \\\"equation_transform\\\",\\n    \\\"equation_numeric_guess\\\": \\\"equation_transform\\\",\\n    \\\"cryptarithm_deduce\\\": \\\"equation_transform\\\",\\n    \\\"cryptarithm_guess\\\": \\\"equation_transform\\\",\\n    \\\"equation_transform\\\": \\\"equation_transform\\\",\\n}\\n\\n\\ndef _normalize_key(value: object) -> str:\\n    text = unicodedata.normalize(\\\"NFKC\\\", str(value or \\\"\\\")).strip().lower()\\n    return re.sub(r\\\"[\\\\s\\\\-]+\\\", \\\"_\\\", text)\\n\\n\\ndef canonical_family(value: object) -> str:\\n    key = _normalize_key(value)\\n    return FAMILY_ALIASES.get(key, key or \\\"unknown\\\")\\n\\n\\ndef classify_puzzle(prompt: str) -> str:\\n    low = str(prompt or \\\"\\\").lower()\\n    if \\\"bit manipulation\\\" in low or \\\"8-bit binary\\\" in low:\\n        return \\\"bit_manipulation\\\"\\n    if \\\"encryption\\\" in low or \\\"decrypt the following text\\\" in low or \\\"cipher\\\" in low:\\n        return \\\"text_encryption\\\"\\n    if \\\"numeral system\\\" in low or \\\"converted into a different numeral\\\" in low:\\n        return \\\"numeral_system\\\"\\n    if \\\"gravitational\\\" in low or \\\"gravity\\\" in low:\\n        return \\\"gravity_constant\\\"\\n    if \\\"transformation rule\\\" in low or \\\"transformation rules\\\" in low:\\n        return \\\"equation_transform\\\"\\n    if \\\"unit conversion\\\" in low or \\\"measurement\\\" in low:\\n        return \\\"unit_conversion\\\"\\n    return \\\"unknown\\\"\\n\\n\\ndef extract_boxed_answers(text: str | None) -> list[str]:\\n    if text is None:\\n        return []\\n    return re.findall(r\\\"\\\\\\\\boxed\\\\{([^}]*)(?:\\\\}|$)\\\", str(text))\\n\\n\\ndef extract_final_answer(text: str | None) -> str:\\n    \\\"\\\"\\\"Extract the final answer with the public Kaggle fallback order.\\\"\\\"\\\"\\n\\n    if text is None:\\n        return \\\"NOT_FOUND\\\"\\n    value = str(text)\\n\\n    matches = extract_boxed_answers(value)\\n    if matches:\\n        non_empty = [match.strip() for match in matches if match.strip()]\\n        if non_empty:\\n            return non_empty[-1]\\n        return matches[-1].strip()\\n\\n    patterns = [\\n        r\\\"The final answer is:\\\\s*([^\\\\n]+)\\\",\\n        r\\\"Final answer is:\\\\s*([^\\\\n]+)\\\",\\n        r\\\"Final answer\\\\s*[:\uff1a]\\\\s*([^\\\\n]+)\\\",\\n        r\\\"final answer\\\\s*[:\uff1a]\\\\s*([^\\\\n]+)\\\",\\n    ]\\n    for pattern in patterns:\\n        matches = re.findall(pattern, value, re.IGNORECASE)\\n        if matches:\\n            return matches[-1].strip()\\n\\n    matches = re.findall(r\\\"-?\\\\d+(?:\\\\.\\\\d+)?\\\", value)\\n    if matches:\\n        return matches[-1]\\n\\n    lines = [line.strip() for line in value.splitlines() if line.strip()]\\n    return lines[-1] if lines else \\\"NOT_FOUND\\\"\\n\\n\\ndef verify_answer(stored_answer: object, predicted: object) -> bool:\\n    \\\"\\\"\\\"Verify a prediction with the public Kaggle metric behavior.\\\"\\\"\\\"\\n\\n    expected = str(stored_answer).strip()\\n    observed = str(predicted).strip()\\n    if re.fullmatch(r\\\"[01]+\\\", expected):\\n        return observed.lower() == expected.lower()\\n    try:\\n        return math.isclose(float(expected), float(observed), rel_tol=1e-2, abs_tol=1e-5)\\n    except Exception:\\n        return observed.lower() == expected.lower()\\n\\n\\ndef canonical_answer(value: object) -> str:\\n    if value is None:\\n        return \\\"\\\"\\n    text = unicodedata.normalize(\\\"NFKC\\\", str(value))\\n    return re.sub(r\\\"\\\\s+\\\", \\\" \\\", text).strip()\\n\\n\\ndef escape_boxed_answer(value: object) -> str:\\n    return str(value).replace(\\\"\\\\\\\\\\\", \\\"\\\\\\\\\\\\\\\\\\\").replace(\\\"{\\\", \\\"\\\\\\\\{\\\").replace(\\\"}\\\", \\\"\\\\\\\\}\\\")\\n\\n\\ndef unescape_latex_braces(value: object) -> str:\\n    return str(value).replace(\\\"\\\\\\\\{\\\", \\\"{\\\").replace(\\\"\\\\\\\\}\\\", \\\"}\\\").replace(\\\"\\\\\\\\\\\\\\\\\\\", \\\"\\\\\\\\\\\")\\n\\n\\ndef canonical_boxed_payload(value: object) -> str:\\n    return canonical_answer(unescape_latex_braces(value))\\n\\n\\ndef parse_finite_number(value: object) -> float | None:\\n    text = canonical_answer(value).replace(\\\",\\\", \\\"\\\")\\n    if not text:\\n        return None\\n    try:\\n        number = float(text)\\n    except ValueError:\\n        return None\\n    return number if math.isfinite(number) else None\\n\\n\\ndef answers_equivalent(\\n    expected: object,\\n    observed: object,\\n    *,\\n    rel_tol: float = 1e-2,\\n    abs_tol: float = 1e-5,\\n    observed_is_boxed_payload: bool = False,\\n) -> bool:\\n    expected_text = canonical_answer(expected)\\n    observed_text = canonical_boxed_payload(observed) if observed_is_boxed_payload else canonical_answer(observed)\\n    expected_number = parse_finite_number(expected_text)\\n    observed_number = parse_finite_number(observed_text)\\n    if expected_number is not None and observed_number is not None:\\n        return math.isclose(expected_number, observed_number, rel_tol=rel_tol, abs_tol=abs_tol)\\n    return expected_text.lower() == observed_text.lower()\\n\\n\\ndef box_answer(value: object) -> str:\\n    return f\\\"\\\\\\\\boxed{{{escape_boxed_answer(value)}}}\\\"\\n\\n\",\n  \"scripts/evaluate_lora_adapter.py\": \"#!/usr/bin/env python3\\n\\\"\\\"\\\"Official-like vLLM evaluator for Nemotron LoRA adapters.\\n\\nThis script is intentionally evaluation-only. It does not train, package, or\\nsubmit. It generates answers with the same scoring-facing settings used by the\\npublic local-CV notebooks: LoRA enabled, max rank 32, 8192 context, 7680 output\\ntokens, deterministic sampling, boxed-answer extraction, and per-family ACC.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport os\\nimport sys\\nimport time\\nfrom collections import defaultdict\\nfrom datetime import datetime, timezone\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport pandas as pd\\n\\nROOT = Path(__file__).resolve().parents[1]\\nif str(ROOT) not in sys.path:\\n    sys.path.insert(0, str(ROOT))\\n\\nfrom src.competition_utils import (  # noqa: E402\\n    MODEL_NAME,\\n    OFFICIAL_INFERENCE_CONFIG,\\n    PROMPT_SUFFIX,\\n    classify_puzzle,\\n    extract_final_answer,\\n    verify_answer,\\n)\\n\\n\\ndef utc_now() -> str:\\n    return datetime.now(timezone.utc).isoformat()\\n\\n\\ndef parse_seeds(raw: str | int | None) -> list[int]:\\n    if raw is None or raw == \\\"\\\":\\n        return [42]\\n    if isinstance(raw, int):\\n        return [raw]\\n    seeds: list[int] = []\\n    for chunk in str(raw).replace(\\\";\\\", \\\",\\\").split(\\\",\\\"):\\n        chunk = chunk.strip()\\n        if chunk:\\n            seeds.append(int(chunk))\\n    return seeds or [42]\\n\\n\\ndef resolve_base_model_path(base_model_path: str = \\\"\\\") -> str:\\n    \\\"\\\"\\\"Resolve the base model path for Colab, Kaggle, or local H100 runs.\\\"\\\"\\\"\\n\\n    if base_model_path:\\n        return base_model_path\\n    env_path = os.environ.get(\\\"KG1_BASE_MODEL_PATH\\\") or os.environ.get(\\\"BASE_MODEL_PATH\\\")\\n    if env_path:\\n        return env_path\\n\\n    kaggle_candidates = [\\n        \\\"/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1\\\",\\n        \\\"/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default\\\",\\n        \\\"/kaggle/input/nemotron-3-nano-30b-a3b-bf16/transformers/default/1\\\",\\n    ]\\n    for candidate in kaggle_candidates:\\n        if Path(candidate).exists():\\n            return candidate\\n    return MODEL_NAME\\n\\n\\ndef resolve_model_revision(base_model_path: str, config: dict[str, Any]) -> str | None:\\n    \\\"\\\"\\\"Use the pinned HF revision for repo IDs, but not for local model paths.\\\"\\\"\\\"\\n\\n    revision = str(config.get(\\\"model_revision\\\") or \\\"\\\").strip()\\n    if not revision:\\n        return None\\n    model_text = str(base_model_path)\\n    if Path(model_text).exists() or os.path.isabs(model_text):\\n        return None\\n    return revision\\n\\n\\ndef row_id_column(frame: pd.DataFrame) -> str:\\n    for candidate in (\\\"id\\\", \\\"row_id\\\"):\\n        if candidate in frame.columns:\\n            return candidate\\n    return str(frame.columns.to_list()[0])\\n\\n\\ndef normalize_questions(solution: pd.DataFrame, questions: pd.DataFrame, limit: int = 0) -> pd.DataFrame:\\n    solution = solution.copy()\\n    questions = questions.copy()\\n    sol_id = row_id_column(solution)\\n    q_id = row_id_column(questions)\\n    if sol_id != \\\"id\\\":\\n        solution = solution.rename(columns={sol_id: \\\"id\\\"})\\n    if q_id != \\\"id\\\":\\n        questions = questions.rename(columns={q_id: \\\"id\\\"})\\n    solution[\\\"id\\\"] = solution[\\\"id\\\"].astype(str)\\n    questions[\\\"id\\\"] = questions[\\\"id\\\"].astype(str)\\n    if \\\"prompt\\\" not in questions.columns:\\n        if \\\"prompt\\\" not in solution.columns:\\n            raise ValueError(\\\"questions or solution must contain a prompt column\\\")\\n        questions = solution[[\\\"id\\\", \\\"prompt\\\"]].copy()\\n    ordered = solution[[\\\"id\\\"]].merge(questions, on=\\\"id\\\", how=\\\"left\\\", validate=\\\"one_to_one\\\")\\n    missing_prompt = ordered[\\\"prompt\\\"].isna().sum()\\n    if missing_prompt:\\n        raise ValueError(f\\\"questions missing prompts for {missing_prompt} solution rows\\\")\\n    if limit > 0:\\n        ordered = ordered.head(limit).copy()\\n    return ordered\\n\\n\\ndef validate_adapter_dir(adapter_dir: str | Path) -> Path:\\n    path = Path(adapter_dir)\\n    if not path.exists():\\n        raise FileNotFoundError(f\\\"adapter path does not exist: {path}\\\")\\n    if path.is_file() and path.suffix == \\\".zip\\\":\\n        raise ValueError(\\\"adapter zip must be extracted before vLLM evaluation\\\")\\n    config = path / \\\"adapter_config.json\\\"\\n    if not config.exists():\\n        raise FileNotFoundError(f\\\"missing adapter_config.json: {config}\\\")\\n    model_files = list(path.glob(\\\"adapter_model.safetensors\\\")) + list(path.glob(\\\"adapter_model.bin\\\"))\\n    if not model_files:\\n        raise FileNotFoundError(f\\\"missing adapter_model.safetensors or adapter_model.bin in {path}\\\")\\n    return path\\n\\n\\ndef render_prompts(tokenizer: Any, questions: pd.DataFrame) -> list[str]:\\n    prompts: list[str] = []\\n    for row in questions.itertuples(index=False):\\n        user_content = str(getattr(row, \\\"prompt\\\")) + PROMPT_SUFFIX\\n        try:\\n            prompt = tokenizer.apply_chat_template(\\n                [{\\\"role\\\": \\\"user\\\", \\\"content\\\": user_content}],\\n                tokenize=False,\\n                add_generation_prompt=True,\\n                enable_thinking=True,\\n            )\\n        except Exception:\\n            prompt = user_content\\n        prompts.append(prompt)\\n    return prompts\\n\\n\\ndef _sampling_params(config: dict[str, Any], seed: int):\\n    from vllm import SamplingParams\\n\\n    kwargs = {\\n        \\\"temperature\\\": float(config.get(\\\"temperature\\\", 0.0)),\\n        \\\"top_p\\\": float(config.get(\\\"top_p\\\", 1.0)),\\n        \\\"max_tokens\\\": int(config.get(\\\"max_tokens\\\", 7680)),\\n    }\\n    try:\\n        return SamplingParams(**kwargs, seed=int(seed))\\n    except TypeError:\\n        return SamplingParams(**kwargs)\\n\\n\\ndef apply_vllm_runtime_safety_settings() -> dict[str, str]:\\n    \\\"\\\"\\\"Apply Colab-safe vLLM settings before importing/initializing vLLM.\\n\\n    Colab H100 runtimes may install vLLM builds where DeepGEMM is enabled by\\n    default, but the `deep_gemm` backend package is absent or too old. In that\\n    state vLLM can fail during engine warmup before any generation starts. The\\n    challenge evaluation does not require DeepGEMM specifically, so disable it\\n    unless the caller explicitly opts back in.\\n    \\\"\\\"\\\"\\n\\n    allow_deep_gemm = os.environ.get(\\\"KG1_ALLOW_VLLM_DEEP_GEMM\\\", \\\"0\\\").strip().lower() in {\\n        \\\"1\\\",\\n        \\\"true\\\",\\n        \\\"yes\\\",\\n        \\\"on\\\",\\n    }\\n    if not allow_deep_gemm:\\n        os.environ[\\\"VLLM_USE_DEEP_GEMM\\\"] = \\\"0\\\"\\n        os.environ[\\\"VLLM_MOE_USE_DEEP_GEMM\\\"] = \\\"0\\\"\\n        os.environ[\\\"VLLM_USE_DEEP_GEMM_E8M0\\\"] = \\\"0\\\"\\n        os.environ[\\\"VLLM_USE_DEEP_GEMM_TMA_ALIGNED_SCALES\\\"] = \\\"0\\\"\\n        os.environ[\\\"VLLM_DEEP_GEMM_WARMUP\\\"] = \\\"skip\\\"\\n    os.environ.setdefault(\\\"VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS\\\", \\\"0\\\")\\n    keys = [\\n        \\\"KG1_ALLOW_VLLM_DEEP_GEMM\\\",\\n        \\\"VLLM_USE_DEEP_GEMM\\\",\\n        \\\"VLLM_MOE_USE_DEEP_GEMM\\\",\\n        \\\"VLLM_USE_DEEP_GEMM_E8M0\\\",\\n        \\\"VLLM_USE_DEEP_GEMM_TMA_ALIGNED_SCALES\\\",\\n        \\\"VLLM_DEEP_GEMM_WARMUP\\\",\\n        \\\"VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS\\\",\\n    ]\\n    return {key: os.environ.get(key, \\\"\\\") for key in keys}\\n\\n\\ndef first_existing_column(frame: pd.DataFrame, candidates: list[str]) -> str | None:\\n    for candidate in candidates:\\n        if candidate in frame.columns:\\n            return candidate\\n    return None\\n\\n\\ndef prepare_merged_predictions(solution: pd.DataFrame, pred: pd.DataFrame) -> pd.DataFrame:\\n    \\\"\\\"\\\"Merge generated rows with labels without relying on pandas suffix names.\\\"\\\"\\\"\\n\\n    pred_for_merge = pred.rename(columns={\\\"prompt\\\": \\\"generated_prompt\\\", \\\"type\\\": \\\"pred_type\\\"}).copy()\\n    merged = solution.merge(pred_for_merge, on=\\\"id\\\", how=\\\"left\\\", validate=\\\"one_to_one\\\")\\n\\n    prompt_col = first_existing_column(merged, [\\\"prompt\\\", \\\"generated_prompt\\\", \\\"prompt_x\\\", \\\"prompt_y\\\"])\\n    if prompt_col is None:\\n        merged[\\\"prompt\\\"] = \\\"\\\"\\n    elif prompt_col != \\\"prompt\\\":\\n        merged[\\\"prompt\\\"] = merged[prompt_col].fillna(\\\"\\\").astype(str)\\n\\n    if \\\"prediction\\\" not in merged.columns:\\n        merged[\\\"prediction\\\"] = \\\"\\\"\\n    if \\\"finish_reason\\\" not in merged.columns:\\n        merged[\\\"finish_reason\\\"] = \\\"\\\"\\n    if \\\"completion_tokens\\\" not in merged.columns:\\n        merged[\\\"completion_tokens\\\"] = 0\\n\\n    type_col = first_existing_column(merged, [\\\"type\\\", \\\"task_type\\\", \\\"family\\\", \\\"type_x\\\", \\\"pred_type\\\", \\\"type_y\\\"])\\n    if type_col is None:\\n        merged[\\\"type\\\"] = merged[\\\"prompt\\\"].map(classify_puzzle)\\n    elif type_col != \\\"type\\\":\\n        merged[\\\"type\\\"] = merged[type_col].fillna(\\\"\\\").astype(str)\\n    missing_type = merged[\\\"type\\\"].fillna(\\\"\\\").astype(str).eq(\\\"\\\")\\n    if missing_type.any():\\n        merged.loc[missing_type, \\\"type\\\"] = merged.loc[missing_type, \\\"prompt\\\"].map(classify_puzzle)\\n\\n    return merged\\n\\n\\ndef evaluate_adapter(\\n    solution: pd.DataFrame,\\n    questions: pd.DataFrame,\\n    *,\\n    lora_path: str,\\n    base_model_path: str,\\n    config: dict[str, Any] | None = None,\\n    seed: int = 42,\\n    raw_predictions_path: str | Path | None = None,\\n) -> tuple[dict[str, Any], pd.DataFrame]:\\n    \\\"\\\"\\\"Run vLLM adapter inference and return a summary plus row predictions.\\\"\\\"\\\"\\n\\n    config = {**OFFICIAL_INFERENCE_CONFIG, **(config or {})}\\n    adapter_dir = validate_adapter_dir(lora_path)\\n    questions = normalize_questions(solution, questions, limit=0)\\n    solution = solution.copy()\\n    id_col = row_id_column(solution)\\n    if id_col != \\\"id\\\":\\n        solution = solution.rename(columns={id_col: \\\"id\\\"})\\n    solution[\\\"id\\\"] = solution[\\\"id\\\"].astype(str)\\n\\n    print(\\\"========================================================================\\\")\\n    print(\\\"KG1 official-like adapter evaluation\\\")\\n    print(\\\"========================================================================\\\")\\n    print(\\\"generated_at_utc =\\\", utc_now())\\n    print(\\\"base_model_path =\\\", base_model_path)\\n    print(\\\"adapter_dir =\\\", adapter_dir)\\n    print(\\\"rows =\\\", len(questions))\\n    print(\\\"seed =\\\", seed)\\n    print(\\\"config =\\\", json.dumps(config, indent=2, sort_keys=True))\\n    print(\\n        \\\"vllm_runtime_safety_settings =\\\",\\n        json.dumps(apply_vllm_runtime_safety_settings(), indent=2, sort_keys=True),\\n    )\\n\\n    from vllm import LLM\\n    from vllm.lora.request import LoRARequest\\n\\n    llm_kwargs = {\\n        \\\"model\\\": str(base_model_path),\\n        \\\"tensor_parallel_size\\\": int(config.get(\\\"tensor_parallel_size\\\", 1)),\\n        \\\"max_num_seqs\\\": int(config.get(\\\"max_num_seqs\\\", 64)),\\n        \\\"gpu_memory_utilization\\\": float(config.get(\\\"gpu_memory_utilization\\\", 0.85)),\\n        \\\"dtype\\\": config.get(\\\"dtype\\\", \\\"auto\\\"),\\n        \\\"max_model_len\\\": int(config.get(\\\"max_model_len\\\", 8192)),\\n        \\\"trust_remote_code\\\": bool(config.get(\\\"trust_remote_code\\\", True)),\\n        \\\"enable_lora\\\": True,\\n        \\\"max_lora_rank\\\": int(config.get(\\\"max_lora_rank\\\", 32)),\\n        \\\"enable_prefix_caching\\\": bool(config.get(\\\"enable_prefix_caching\\\", True)),\\n        \\\"enable_chunked_prefill\\\": bool(config.get(\\\"enable_chunked_prefill\\\", True)),\\n    }\\n    model_revision = resolve_model_revision(str(base_model_path), config)\\n    if model_revision:\\n        llm_kwargs[\\\"revision\\\"] = model_revision\\n        llm_kwargs[\\\"tokenizer_revision\\\"] = model_revision\\n    print(\\\"llm_revision =\\\", llm_kwargs.get(\\\"revision\\\", \\\"local_path_or_default\\\"))\\n    if config.get(\\\"enforce_eager\\\") is not None:\\n        llm_kwargs[\\\"enforce_eager\\\"] = bool(config[\\\"enforce_eager\\\"])\\n\\n    start = time.time()\\n    llm = LLM(**llm_kwargs)\\n    tokenizer = llm.get_tokenizer()\\n    print(f\\\"vLLM loaded in {time.time() - start:.1f}s\\\")\\n\\n    rendered = render_prompts(tokenizer, questions)\\n    sampling_params = _sampling_params(config, seed)\\n    lora_request = LoRARequest(\\\"adapter\\\", 1, str(adapter_dir))\\n\\n    if rendered:\\n        warmup_n = min(4, len(rendered))\\n        print(f\\\"warmup_rows = {warmup_n}\\\")\\n        warmup_start = time.time()\\n        _ = llm.generate(rendered[:warmup_n], sampling_params=sampling_params, lora_request=lora_request)\\n        print(f\\\"warmup_elapsed_s = {time.time() - warmup_start:.1f}\\\")\\n\\n    gen_start = time.time()\\n    outputs = llm.generate(rendered, sampling_params=sampling_params, lora_request=lora_request)\\n    gen_elapsed = time.time() - gen_start\\n    print(f\\\"generation_elapsed_s = {gen_elapsed:.1f}\\\")\\n\\n    rows: list[dict[str, Any]] = []\\n    for row, output in zip(questions.itertuples(index=False), outputs):\\n        completion = output.outputs[0]\\n        raw_output = completion.text\\n        prediction = extract_final_answer(raw_output)\\n        row_id = str(getattr(row, \\\"id\\\"))\\n        prompt = str(getattr(row, \\\"prompt\\\"))\\n        rows.append(\\n            {\\n                \\\"id\\\": row_id,\\n                \\\"prompt\\\": prompt,\\n                \\\"raw_output\\\": raw_output,\\n                \\\"prediction\\\": prediction,\\n                \\\"prompt_tokens\\\": len(getattr(output, \\\"prompt_token_ids\\\", []) or []),\\n                \\\"completion_tokens\\\": len(getattr(completion, \\\"token_ids\\\", []) or []),\\n                \\\"finish_reason\\\": completion.finish_reason or \\\"\\\",\\n                \\\"type\\\": classify_puzzle(prompt),\\n            }\\n        )\\n\\n    pred = pd.DataFrame(rows)\\n    if raw_predictions_path is not None:\\n        raw_path = Path(raw_predictions_path)\\n        raw_path.parent.mkdir(parents=True, exist_ok=True)\\n        pred.to_csv(raw_path, index=False)\\n        print(\\\"raw_predictions_pre_score_csv =\\\", raw_path)\\n        print(\\\"raw_predictions_pre_score_rows =\\\", len(pred))\\n\\n    merged = prepare_merged_predictions(solution, pred)\\n    if \\\"answer\\\" in merged.columns:\\n        merged[\\\"correct\\\"] = merged.apply(lambda r: verify_answer(r[\\\"answer\\\"], r[\\\"prediction\\\"]), axis=1)\\n    else:\\n        merged[\\\"correct\\\"] = False\\n    merged[\\\"truncated\\\"] = merged[\\\"finish_reason\\\"].fillna(\\\"\\\").astype(str).eq(\\\"length\\\")\\n\\n    total_tokens = int(merged[\\\"completion_tokens\\\"].fillna(0).sum())\\n    summary = {\\n        \\\"generated_at_utc\\\": utc_now(),\\n        \\\"base_model_path\\\": str(base_model_path),\\n        \\\"adapter_dir\\\": str(adapter_dir),\\n        \\\"rows\\\": int(len(merged)),\\n        \\\"correct\\\": int(merged[\\\"correct\\\"].sum()),\\n        \\\"accuracy\\\": float(merged[\\\"correct\\\"].mean()) if len(merged) else 0.0,\\n        \\\"truncated\\\": int(merged[\\\"truncated\\\"].sum()),\\n        \\\"truncation_rate\\\": float(merged[\\\"truncated\\\"].mean()) if len(merged) else 0.0,\\n        \\\"completion_tokens\\\": total_tokens,\\n        \\\"generation_elapsed_s\\\": gen_elapsed,\\n        \\\"tokens_per_second\\\": float(total_tokens / gen_elapsed) if gen_elapsed > 0 else 0.0,\\n        \\\"seed\\\": int(seed),\\n        \\\"config\\\": config,\\n    }\\n    print(\\\"summary =\\\", json.dumps(summary, indent=2, sort_keys=True))\\n    return summary, merged\\n\\n\\ndef summarize_per_task(frame: pd.DataFrame) -> pd.DataFrame:\\n    rows: list[dict[str, Any]] = []\\n    grouped = frame.groupby(\\\"type\\\", dropna=False)\\n    for family, group in grouped:\\n        total = int(len(group))\\n        correct = int(group[\\\"correct\\\"].sum())\\n        truncated = int(group[\\\"truncated\\\"].sum()) if \\\"truncated\\\" in group else 0\\n        rows.append(\\n            {\\n                \\\"task_type\\\": str(family),\\n                \\\"total\\\": total,\\n                \\\"correct\\\": correct,\\n                \\\"accuracy\\\": correct / total if total else 0.0,\\n                \\\"truncated\\\": truncated,\\n                \\\"truncation_rate\\\": truncated / total if total else 0.0,\\n            }\\n        )\\n    total = int(len(frame))\\n    correct = int(frame[\\\"correct\\\"].sum()) if \\\"correct\\\" in frame else 0\\n    truncated = int(frame[\\\"truncated\\\"].sum()) if \\\"truncated\\\" in frame else 0\\n    rows.append(\\n        {\\n            \\\"task_type\\\": \\\"OVERALL\\\",\\n            \\\"total\\\": total,\\n            \\\"correct\\\": correct,\\n            \\\"accuracy\\\": correct / total if total else 0.0,\\n            \\\"truncated\\\": truncated,\\n            \\\"truncation_rate\\\": truncated / total if total else 0.0,\\n        }\\n    )\\n    return pd.DataFrame(rows)\\n\\n\\ndef main() -> int:\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\\"--solution-csv\\\", type=Path, required=True)\\n    parser.add_argument(\\\"--questions-csv\\\", type=Path, default=None)\\n    parser.add_argument(\\\"--adapter\\\", type=Path, required=True)\\n    parser.add_argument(\\\"--base-model-path\\\", default=\\\"\\\")\\n    parser.add_argument(\\\"--label\\\", default=\\\"adapter\\\")\\n    parser.add_argument(\\\"--seed\\\", type=int, default=42)\\n    parser.add_argument(\\\"--limit\\\", type=int, default=0)\\n    parser.add_argument(\\\"--output-dir\\\", type=Path, required=True)\\n    args = parser.parse_args()\\n\\n    args.output_dir.mkdir(parents=True, exist_ok=True)\\n    solution = pd.read_csv(args.solution_csv)\\n    if args.limit > 0:\\n        solution = solution.head(args.limit).copy()\\n    questions = pd.read_csv(args.questions_csv or args.solution_csv)\\n    if args.limit > 0:\\n        ids = set(solution[row_id_column(solution)].astype(str))\\n        q_id = row_id_column(questions)\\n        questions = questions[questions[q_id].astype(str).isin(ids)].copy()\\n\\n    label = args.label.replace(\\\"/\\\", \\\"_\\\").replace(\\\"\\\\\\\\\\\", \\\"_\\\")\\n    raw_predictions_path = args.output_dir / f\\\"{label}_raw_predictions_pre_score.csv\\\"\\n    summary, predictions = evaluate_adapter(\\n        solution,\\n        questions,\\n        lora_path=str(args.adapter),\\n        base_model_path=resolve_base_model_path(args.base_model_path),\\n        config=OFFICIAL_INFERENCE_CONFIG,\\n        seed=args.seed,\\n        raw_predictions_path=raw_predictions_path,\\n    )\\n    predictions_path = args.output_dir / f\\\"{label}_predictions.csv\\\"\\n    per_task_path = args.output_dir / f\\\"{label}_per_task.csv\\\"\\n    report_path = args.output_dir / f\\\"{label}_eval_report.json\\\"\\n    predictions.to_csv(predictions_path, index=False)\\n    summarize_per_task(predictions).to_csv(per_task_path, index=False)\\n    report = {\\n        **summary,\\n        \\\"label\\\": args.label,\\n        \\\"inputs\\\": {\\n            \\\"solution_csv\\\": str(args.solution_csv),\\n            \\\"questions_csv\\\": str(args.questions_csv or args.solution_csv),\\n            \\\"adapter\\\": str(args.adapter),\\n            \\\"limit\\\": args.limit,\\n        },\\n        \\\"outputs\\\": {\\n            \\\"raw_predictions_pre_score_csv\\\": str(raw_predictions_path),\\n            \\\"predictions_csv\\\": str(predictions_path),\\n            \\\"per_task_csv\\\": str(per_task_path),\\n            \\\"report_json\\\": str(report_path),\\n        },\\n    }\\n    report_path.write_text(json.dumps(report, indent=2, sort_keys=True), encoding=\\\"utf-8\\\")\\n    print(\\\"predictions_csv =\\\", predictions_path)\\n    print(\\\"per_task_csv =\\\", per_task_path)\\n    print(\\\"report_json =\\\", report_path)\\n    return 0\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    raise SystemExit(main())\\n\",\n  \"scripts/solve_rate_gate.py\": \"#!/usr/bin/env python3\\n\\\"\\\"\\\"Solve-rate promotion gate for Nemotron LoRA candidates.\\n\\nThis is the score-facing gate: compare candidate vs baseline by decoded\\nanswers, official answer verification, and per-family regressions. It supports\\ntwo modes:\\n\\n1. CSV mode: compare existing prediction CSVs.\\n2. Adapter mode: run vLLM evaluation for baseline and candidate adapters.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport statistics\\nimport sys\\nfrom datetime import datetime, timezone\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport pandas as pd\\n\\nROOT = Path(__file__).resolve().parents[1]\\nif str(ROOT) not in sys.path:\\n    sys.path.insert(0, str(ROOT))\\n\\nfrom scripts.evaluate_lora_adapter import evaluate_adapter, parse_seeds, resolve_base_model_path\\nfrom src.competition_utils import (\\n    OFFICIAL_INFERENCE_CONFIG,\\n    classify_puzzle,\\n    extract_boxed_answers,\\n    extract_final_answer,\\n    verify_answer,\\n)\\n\\n\\ndef utc_now() -> str:\\n    return datetime.now(timezone.utc).isoformat()\\n\\n\\ndef row_id_column(frame: pd.DataFrame) -> str:\\n    for candidate in (\\\"id\\\", \\\"row_id\\\"):\\n        if candidate in frame.columns:\\n            return candidate\\n    return str(frame.columns.to_list()[0])\\n\\n\\ndef family_for_row(row: pd.Series) -> str:\\n    for key in (\\\"type\\\", \\\"family\\\", \\\"task_family\\\"):\\n        value = row.get(key)\\n        if value not in (None, \\\"\\\"):\\n            return str(value)\\n    return classify_puzzle(str(row.get(\\\"prompt\\\", \\\"\\\")))\\n\\n\\ndef normalize_solution(solution_csv: Path, limit: int = 0) -> pd.DataFrame:\\n    solution = pd.read_csv(solution_csv)\\n    required = {\\\"prompt\\\", \\\"answer\\\"}\\n    missing = sorted(required - set(solution.columns))\\n    if missing:\\n        raise ValueError(f\\\"solution CSV missing required columns: {missing}\\\")\\n    id_col = row_id_column(solution)\\n    if id_col != \\\"id\\\":\\n        solution = solution.rename(columns={id_col: \\\"id\\\"})\\n    solution[\\\"id\\\"] = solution[\\\"id\\\"].astype(str)\\n    solution[\\\"family_gate\\\"] = solution.apply(family_for_row, axis=1)\\n    if limit > 0:\\n        solution = solution.head(limit).copy()\\n    return solution\\n\\n\\ndef predictions_from_csv(predictions_csv: Path, label: str) -> pd.DataFrame:\\n    predictions = pd.read_csv(predictions_csv)\\n    id_col = row_id_column(predictions)\\n    if id_col != \\\"id\\\":\\n        predictions = predictions.rename(columns={id_col: \\\"id\\\"})\\n    predictions[\\\"id\\\"] = predictions[\\\"id\\\"].astype(str)\\n\\n    if \\\"raw_output\\\" not in predictions.columns and \\\"prediction\\\" not in predictions.columns:\\n        raise ValueError(f\\\"{label} predictions need a 'prediction' or 'raw_output' column\\\")\\n    if \\\"raw_output\\\" not in predictions.columns:\\n        predictions[\\\"raw_output\\\"] = predictions[\\\"prediction\\\"].astype(str)\\n    if \\\"prediction\\\" not in predictions.columns:\\n        predictions[\\\"prediction\\\"] = predictions[\\\"raw_output\\\"].map(extract_final_answer)\\n    else:\\n        missing_prediction = predictions[\\\"prediction\\\"].isna() | (predictions[\\\"prediction\\\"].astype(str) == \\\"\\\")\\n        predictions.loc[missing_prediction, \\\"prediction\\\"] = predictions.loc[\\n            missing_prediction, \\\"raw_output\\\"\\n        ].map(extract_final_answer)\\n    return predictions[[\\\"id\\\", \\\"prediction\\\", \\\"raw_output\\\"]].copy()\\n\\n\\ndef score_predictions(solution: pd.DataFrame, predictions: pd.DataFrame, label: str, seed: int | None = None) -> pd.DataFrame:\\n    merged = solution.merge(predictions, on=\\\"id\\\", how=\\\"left\\\", validate=\\\"one_to_one\\\")\\n    merged[\\\"label\\\"] = label\\n    merged[\\\"seed\\\"] = seed if seed is not None else 0\\n    merged[\\\"prediction\\\"] = merged[\\\"prediction\\\"].fillna(\\\"NOT_FOUND\\\").astype(str)\\n    merged[\\\"raw_output\\\"] = merged[\\\"raw_output\\\"].fillna(\\\"\\\").astype(str)\\n    merged[\\\"final_answer\\\"] = merged.apply(\\n        lambda row: extract_final_answer(row[\\\"raw_output\\\"]) if row[\\\"raw_output\\\"] else row[\\\"prediction\\\"],\\n        axis=1,\\n    )\\n    merged[\\\"boxed_valid\\\"] = merged[\\\"raw_output\\\"].map(lambda value: len(extract_boxed_answers(value)) > 0)\\n    merged[\\\"correct\\\"] = merged.apply(\\n        lambda row: verify_answer(str(row[\\\"answer\\\"]), str(row[\\\"final_answer\\\"])),\\n        axis=1,\\n    )\\n    return merged\\n\\n\\ndef prediction_frames_from_adapters(\\n    solution: pd.DataFrame,\\n    questions: pd.DataFrame,\\n    *,\\n    baseline_adapter: Path,\\n    candidate_adapter: Path,\\n    base_model_path: str,\\n    seeds: list[int],\\n) -> tuple[pd.DataFrame, pd.DataFrame]:\\n    baseline_frames: list[pd.DataFrame] = []\\n    candidate_frames: list[pd.DataFrame] = []\\n    for seed in seeds:\\n        _, baseline_merged = evaluate_adapter(\\n            solution,\\n            questions,\\n            lora_path=str(baseline_adapter),\\n            base_model_path=base_model_path,\\n            config=OFFICIAL_INFERENCE_CONFIG,\\n            seed=seed,\\n        )\\n        _, candidate_merged = evaluate_adapter(\\n            solution,\\n            questions,\\n            lora_path=str(candidate_adapter),\\n            base_model_path=base_model_path,\\n            config=OFFICIAL_INFERENCE_CONFIG,\\n            seed=seed,\\n        )\\n        baseline_merged = baseline_merged.rename(columns={\\\"type\\\": \\\"family_gate\\\"})\\n        candidate_merged = candidate_merged.rename(columns={\\\"type\\\": \\\"family_gate\\\"})\\n        baseline_frames.append(score_predictions(solution, baseline_merged[[\\\"id\\\", \\\"prediction\\\", \\\"raw_output\\\"]], \\\"baseline\\\", seed))\\n        candidate_frames.append(score_predictions(solution, candidate_merged[[\\\"id\\\", \\\"prediction\\\", \\\"raw_output\\\"]], \\\"candidate\\\", seed))\\n    return pd.concat(baseline_frames, ignore_index=True), pd.concat(candidate_frames, ignore_index=True)\\n\\n\\ndef summarize(frame: pd.DataFrame) -> dict[str, Any]:\\n    by_family: dict[str, dict[str, Any]] = {}\\n    for family, group in frame.groupby(\\\"family_gate\\\"):\\n        by_family[str(family)] = {\\n            \\\"rows\\\": int(len(group)),\\n            \\\"correct\\\": int(group[\\\"correct\\\"].sum()),\\n            \\\"accuracy\\\": float(group[\\\"correct\\\"].mean()) if len(group) else 0.0,\\n            \\\"boxed_format_rate\\\": float(group[\\\"boxed_valid\\\"].mean()) if len(group) else 0.0,\\n        }\\n\\n    seed_summaries = []\\n    for seed, group in frame.groupby(\\\"seed\\\"):\\n        seed_summaries.append(\\n            {\\n                \\\"seed\\\": int(seed),\\n                \\\"rows\\\": int(len(group)),\\n                \\\"correct\\\": int(group[\\\"correct\\\"].sum()),\\n                \\\"accuracy\\\": float(group[\\\"correct\\\"].mean()) if len(group) else 0.0,\\n                \\\"boxed_format_rate\\\": float(group[\\\"boxed_valid\\\"].mean()) if len(group) else 0.0,\\n            }\\n        )\\n    accuracies = [item[\\\"accuracy\\\"] for item in seed_summaries]\\n    return {\\n        \\\"rows\\\": int(len(frame)),\\n        \\\"correct\\\": int(frame[\\\"correct\\\"].sum()),\\n        \\\"accuracy\\\": float(frame[\\\"correct\\\"].mean()) if len(frame) else 0.0,\\n        \\\"boxed_format_rate\\\": float(frame[\\\"boxed_valid\\\"].mean()) if len(frame) else 0.0,\\n        \\\"accuracy_min_seed\\\": float(min(accuracies)) if accuracies else 0.0,\\n        \\\"accuracy_mean_seed\\\": float(statistics.mean(accuracies)) if accuracies else 0.0,\\n        \\\"accuracy_max_seed\\\": float(max(accuracies)) if accuracies else 0.0,\\n        \\\"by_seed\\\": seed_summaries,\\n        \\\"by_family\\\": by_family,\\n    }\\n\\n\\ndef compare(\\n    baseline: pd.DataFrame,\\n    candidate: pd.DataFrame,\\n    *,\\n    family_regression_tolerance: float,\\n    min_net_gain: float,\\n    min_boxed_rate: float,\\n) -> tuple[bool, list[str], dict[str, Any]]:\\n    baseline_summary = summarize(baseline)\\n    candidate_summary = summarize(candidate)\\n    reasons: list[str] = []\\n\\n    baseline_accuracy = float(baseline_summary[\\\"accuracy\\\"])\\n    candidate_accuracy = float(candidate_summary[\\\"accuracy\\\"])\\n    net_gain = candidate_accuracy - baseline_accuracy\\n    if net_gain <= min_net_gain:\\n        reasons.append(f\\\"net_gain_not_above_threshold:{net_gain:.6f}<={min_net_gain:.6f}\\\")\\n\\n    if float(candidate_summary[\\\"boxed_format_rate\\\"]) < min_boxed_rate:\\n        reasons.append(\\n            \\\"candidate_boxed_rate_below_threshold:\\\"\\n            f\\\"{candidate_summary['boxed_format_rate']:.6f}<{min_boxed_rate:.6f}\\\"\\n        )\\n\\n    baseline_families = baseline_summary[\\\"by_family\\\"]\\n    candidate_families = candidate_summary[\\\"by_family\\\"]\\n    family_deltas: dict[str, dict[str, float]] = {}\\n    for family in sorted(set(baseline_families) | set(candidate_families)):\\n        b_acc = float(baseline_families.get(family, {}).get(\\\"accuracy\\\", 0.0))\\n        c_acc = float(candidate_families.get(family, {}).get(\\\"accuracy\\\", 0.0))\\n        delta = c_acc - b_acc\\n        family_deltas[family] = {\\n            \\\"baseline_accuracy\\\": b_acc,\\n            \\\"candidate_accuracy\\\": c_acc,\\n            \\\"delta\\\": delta,\\n        }\\n        if delta < -family_regression_tolerance:\\n            reasons.append(\\n                f\\\"family_regression:{family}:{c_acc:.6f}<{b_acc:.6f}-\\\"\\n                f\\\"{family_regression_tolerance:.6f}\\\"\\n            )\\n\\n    comparison = {\\n        \\\"baseline\\\": baseline_summary,\\n        \\\"candidate\\\": candidate_summary,\\n        \\\"net_gain\\\": net_gain,\\n        \\\"family_deltas\\\": family_deltas,\\n    }\\n    return len(reasons) == 0, reasons, comparison\\n\\n\\ndef write_failures(path: Path, baseline: pd.DataFrame, candidate: pd.DataFrame) -> None:\\n    cols = [\\\"id\\\", \\\"family_gate\\\", \\\"answer\\\", \\\"final_answer\\\", \\\"correct\\\"]\\n    b = baseline[cols].rename(columns={\\\"final_answer\\\": \\\"baseline_final_answer\\\", \\\"correct\\\": \\\"baseline_correct\\\"})\\n    c = candidate[cols].rename(columns={\\\"final_answer\\\": \\\"candidate_final_answer\\\", \\\"correct\\\": \\\"candidate_correct\\\"})\\n    merged = b.merge(c, on=[\\\"id\\\", \\\"family_gate\\\", \\\"answer\\\"], how=\\\"outer\\\")\\n    merged[\\\"regressed\\\"] = merged[\\\"baseline_correct\\\"].fillna(False) & ~merged[\\\"candidate_correct\\\"].fillna(False)\\n    merged[\\\"improved\\\"] = ~merged[\\\"baseline_correct\\\"].fillna(False) & merged[\\\"candidate_correct\\\"].fillna(False)\\n    merged.to_csv(path, index=False)\\n\\n\\ndef run_self_test(solution_csv: Path, output_dir: Path, limit: int) -> int:\\n    solution = normalize_solution(solution_csv, limit=limit or 20)\\n    baseline_predictions = pd.DataFrame(\\n        {\\n            \\\"id\\\": solution[\\\"id\\\"],\\n            \\\"prediction\\\": solution[\\\"answer\\\"].astype(str),\\n            \\\"raw_output\\\": \\\"\\\\\\\\boxed{\\\" + solution[\\\"answer\\\"].astype(str) + \\\"}\\\",\\n        }\\n    )\\n    candidate_predictions = baseline_predictions.copy()\\n    if len(candidate_predictions):\\n        candidate_predictions.loc[candidate_predictions.index[-1], \\\"prediction\\\"] = \\\"INTENTIONAL_WRONG\\\"\\n        candidate_predictions.loc[candidate_predictions.index[-1], \\\"raw_output\\\"] = \\\"\\\\\\\\boxed{INTENTIONAL_WRONG}\\\"\\n    baseline = score_predictions(solution, baseline_predictions, \\\"baseline\\\")\\n    candidate = score_predictions(solution, candidate_predictions, \\\"candidate\\\")\\n    approved, reasons, comparison = compare(\\n        baseline,\\n        candidate,\\n        family_regression_tolerance=0.0,\\n        min_net_gain=0.0,\\n        min_boxed_rate=1.0,\\n    )\\n    output_dir.mkdir(parents=True, exist_ok=True)\\n    payload = {\\n        \\\"self_test\\\": True,\\n        \\\"approved\\\": approved,\\n        \\\"reasons\\\": reasons,\\n        \\\"comparison\\\": comparison,\\n    }\\n    (output_dir / \\\"self_test_report.json\\\").write_text(json.dumps(payload, indent=2), encoding=\\\"utf-8\\\")\\n    write_failures(output_dir / \\\"self_test_row_deltas.csv\\\", baseline, candidate)\\n    print(json.dumps(payload, indent=2))\\n    return 0 if not approved and reasons else 2\\n\\n\\ndef main() -> int:\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\\"--solution-csv\\\", type=Path, default=ROOT / \\\"data\\\" / \\\"splits\\\" / \\\"val_public_proxy.csv\\\")\\n    parser.add_argument(\\\"--questions-csv\\\", type=Path, default=None)\\n    parser.add_argument(\\\"--baseline-predictions\\\", type=Path, default=None)\\n    parser.add_argument(\\\"--candidate-predictions\\\", type=Path, default=None)\\n    parser.add_argument(\\\"--baseline-adapter\\\", type=Path, default=None)\\n    parser.add_argument(\\\"--candidate-adapter\\\", type=Path, default=None)\\n    parser.add_argument(\\\"--base-model-path\\\", default=\\\"\\\")\\n    parser.add_argument(\\\"--seeds\\\", default=\\\"42\\\")\\n    parser.add_argument(\\\"--limit\\\", type=int, default=0)\\n    parser.add_argument(\\\"--family-regression-tolerance\\\", type=float, default=0.0)\\n    parser.add_argument(\\\"--min-net-gain\\\", type=float, default=0.0)\\n    parser.add_argument(\\\"--min-boxed-rate\\\", type=float, default=0.98)\\n    parser.add_argument(\\\"--output-dir\\\", type=Path, default=ROOT / \\\"artifacts\\\" / \\\"solve_rate_gate\\\")\\n    parser.add_argument(\\\"--json-output\\\", type=Path, default=None)\\n    parser.add_argument(\\\"--self-test\\\", action=\\\"store_true\\\")\\n    args = parser.parse_args()\\n\\n    if args.self_test:\\n        return run_self_test(args.solution_csv, args.output_dir, args.limit)\\n\\n    solution = normalize_solution(args.solution_csv, args.limit)\\n    questions_csv = args.questions_csv or args.solution_csv\\n    questions = pd.read_csv(questions_csv)\\n    id_col = row_id_column(questions)\\n    if id_col != \\\"id\\\":\\n        questions = questions.rename(columns={id_col: \\\"id\\\"})\\n    questions[\\\"id\\\"] = questions[\\\"id\\\"].astype(str)\\n    if args.limit > 0:\\n        questions = questions[questions[\\\"id\\\"].isin(set(solution[\\\"id\\\"]))].copy()\\n\\n    csv_mode = args.baseline_predictions is not None and args.candidate_predictions is not None\\n    adapter_mode = args.baseline_adapter is not None and args.candidate_adapter is not None\\n    if csv_mode == adapter_mode:\\n        raise SystemExit(\\\"Choose exactly one mode: prediction CSVs or adapter paths.\\\")\\n\\n    if csv_mode:\\n        baseline = score_predictions(\\n            solution,\\n            predictions_from_csv(args.baseline_predictions, \\\"baseline\\\"),  # type: ignore[arg-type]\\n            \\\"baseline\\\",\\n        )\\n        candidate = score_predictions(\\n            solution,\\n            predictions_from_csv(args.candidate_predictions, \\\"candidate\\\"),  # type: ignore[arg-type]\\n            \\\"candidate\\\",\\n        )\\n    else:\\n        baseline, candidate = prediction_frames_from_adapters(\\n            solution,\\n            questions,\\n            baseline_adapter=args.baseline_adapter,  # type: ignore[arg-type]\\n            candidate_adapter=args.candidate_adapter,  # type: ignore[arg-type]\\n            base_model_path=resolve_base_model_path(args.base_model_path),\\n            seeds=parse_seeds(args.seeds),\\n        )\\n\\n    approved, reasons, comparison = compare(\\n        baseline,\\n        candidate,\\n        family_regression_tolerance=args.family_regression_tolerance,\\n        min_net_gain=args.min_net_gain,\\n        min_boxed_rate=args.min_boxed_rate,\\n    )\\n    args.output_dir.mkdir(parents=True, exist_ok=True)\\n    json_output = args.json_output or args.output_dir / \\\"solve_rate_gate_report.json\\\"\\n    report = {\\n        \\\"generated_at_utc\\\": utc_now(),\\n        \\\"status\\\": \\\"approve\\\" if approved else \\\"reject\\\",\\n        \\\"approved\\\": approved,\\n        \\\"reasons\\\": reasons,\\n        \\\"thresholds\\\": {\\n            \\\"family_regression_tolerance\\\": args.family_regression_tolerance,\\n            \\\"min_net_gain\\\": args.min_net_gain,\\n            \\\"min_boxed_rate\\\": args.min_boxed_rate,\\n        },\\n        \\\"inputs\\\": {\\n            \\\"solution_csv\\\": str(args.solution_csv),\\n            \\\"questions_csv\\\": str(questions_csv),\\n            \\\"baseline_predictions\\\": str(args.baseline_predictions) if args.baseline_predictions else \\\"\\\",\\n            \\\"candidate_predictions\\\": str(args.candidate_predictions) if args.candidate_predictions else \\\"\\\",\\n            \\\"baseline_adapter\\\": str(args.baseline_adapter) if args.baseline_adapter else \\\"\\\",\\n            \\\"candidate_adapter\\\": str(args.candidate_adapter) if args.candidate_adapter else \\\"\\\",\\n            \\\"seeds\\\": parse_seeds(args.seeds),\\n            \\\"limit\\\": args.limit,\\n        },\\n        \\\"comparison\\\": comparison,\\n    }\\n    json_output.write_text(json.dumps(report, indent=2), encoding=\\\"utf-8\\\")\\n    write_failures(args.output_dir / \\\"solve_rate_row_deltas.csv\\\", baseline, candidate)\\n    print(json.dumps(report, indent=2))\\n    return 0 if approved else 2\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    raise SystemExit(main())\\n\"\n}")
for rel, content in FILES.items():
    path = ROOT / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
    print('wrote', path, 'bytes=', path.stat().st_size)
for rel in ['src/competition_utils.py', 'scripts/evaluate_lora_adapter.py', 'scripts/solve_rate_gate.py']:
    py_compile.compile(str(ROOT / rel), doraise=True)
    print('compiled', rel)
print('=== V207B SCRIPT BOOTSTRAP END ===')


In [ ]:
# CELL: verify or bootstrap V207A baseline artifacts and build weak-family CSV.
print('=== V207B V207A ARTIFACT CHECK START ===')
import urllib.request
import pandas as pd
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.competition_utils import classify_puzzle

FALLBACK_EXPORTS = {
    BASELINE_PREDICTIONS: 'v194_baseline_predictions.csv',
    BASELINE_PER_TASK: 'v194_baseline_per_task.csv',
    BASELINE_REPORT: 'v194_baseline_eval_report.json',
}

def download_fallback_export(dst, filename):
    dst = pathlib.Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    url = FALLBACK_EXPORT_BASE.rstrip('/') + '/' + filename
    tmp = dst.with_suffix(dst.suffix + '.tmp')
    print('fallback_download_url =', url)
    print('fallback_download_dst =', dst)
    urllib.request.urlretrieve(url, tmp)
    tmp.replace(dst)
    print('fallback_downloaded_bytes =', dst.stat().st_size)

def ensure_fallback_file(path, filename):
    path = pathlib.Path(path)
    print('checking', path, 'exists=', path.exists())
    if not path.exists():
        print('missing artifact; downloading validated fallback export:', filename)
        download_fallback_export(path, filename)
    print('artifact_ready =', path, 'bytes=', path.stat().st_size)

for path, filename in FALLBACK_EXPORTS.items():
    ensure_fallback_file(path, filename)

if not VAL_CSV.exists():
    print('VAL_CSV missing; reconstructing validation CSV from baseline predictions export.')
    pred = pd.read_csv(BASELINE_PREDICTIONS)
    prompt_col = next((c for c in ['prompt', 'prompt_x', 'prompt_y'] if c in pred.columns), None)
    family_source_col = next((c for c in ['family', 'type'] if c in pred.columns), None)
    required = {'id', 'answer'}
    missing = sorted(required - set(pred.columns))
    if missing or prompt_col is None:
        raise RuntimeError(
            f'Baseline predictions fallback cannot reconstruct validation CSV. '
            f'missing={missing}, prompt_col={prompt_col}, columns={list(pred.columns)}'
        )
    val_cols = ['id', prompt_col, 'answer']
    if family_source_col:
        val_cols.append(family_source_col)
    val = pred[val_cols].copy()
    val = val.rename(columns={prompt_col: 'prompt'})
    if family_source_col and family_source_col != 'family':
        val = val.rename(columns={family_source_col: 'family'})
    if 'family' not in val.columns:
        val['family'] = val['prompt'].map(classify_puzzle)
    val = val[['id', 'prompt', 'answer', 'family']]
    val.to_csv(VAL_CSV, index=False)
    print('VAL_CSV reconstructed rows =', len(val))
    print('validation_source = v194_baseline_predictions_drive_export_fallback')
else:
    print('VAL_CSV already exists:', VAL_CSV)

required_paths = [VAL_CSV, BASELINE_PREDICTIONS, BASELINE_PER_TASK, BASELINE_REPORT]
for path in required_paths:
    print('final_check', path, 'exists=', path.exists(), 'bytes=', path.stat().st_size if path.exists() else None)
    if not path.exists():
        raise FileNotFoundError('Missing required V207B artifact after fallback bootstrap: ' + str(path))

val = pd.read_csv(VAL_CSV)
baseline_pred_audit = pd.read_csv(BASELINE_PREDICTIONS)
baseline_per_audit = pd.read_csv(BASELINE_PER_TASK)
baseline_report_audit = json.loads(BASELINE_REPORT.read_text(encoding='utf-8'))

required_val_cols = {'id', 'prompt', 'answer'}
required_pred_cols = {'id', 'answer', 'prediction', 'raw_output', 'correct', 'truncated'}
required_per_cols = {'task_type', 'total', 'correct', 'accuracy', 'truncated', 'truncation_rate'}
missing_val_cols = sorted(required_val_cols - set(val.columns))
missing_pred_cols = sorted(required_pred_cols - set(baseline_pred_audit.columns))
missing_per_cols = sorted(required_per_cols - set(baseline_per_audit.columns))
print('baseline_artifact_audit_val_rows =', len(val), 'missing_cols=', missing_val_cols)
print('baseline_artifact_audit_predictions_rows =', len(baseline_pred_audit), 'missing_cols=', missing_pred_cols)
print('baseline_artifact_audit_per_task_rows =', len(baseline_per_audit), 'missing_cols=', missing_per_cols)
print(
    'baseline_artifact_audit_report =',
    json.dumps(
        {
            'rows': baseline_report_audit.get('rows'),
            'correct': baseline_report_audit.get('correct'),
            'accuracy': baseline_report_audit.get('accuracy'),
            'truncated': baseline_report_audit.get('truncated'),
        },
        sort_keys=True,
    ),
)
if missing_val_cols or missing_pred_cols or missing_per_cols:
    raise RuntimeError(
        'Baseline fallback artifacts have invalid schema: '
        f'val={missing_val_cols}, predictions={missing_pred_cols}, per_task={missing_per_cols}'
    )
if len(val) != 947 or len(baseline_pred_audit) != 947:
    raise RuntimeError(f'Expected 947 validation/baseline rows, got val={len(val)} pred={len(baseline_pred_audit)}')
if not val['id'].astype(str).is_unique or not baseline_pred_audit['id'].astype(str).is_unique:
    raise RuntimeError('Validation or baseline prediction IDs are not unique.')
val_ids = set(val['id'].astype(str))
pred_ids = set(baseline_pred_audit['id'].astype(str))
if val_ids != pred_ids:
    raise RuntimeError(f'Validation and baseline prediction ID sets differ: val={len(val_ids)} pred={len(pred_ids)}')
baseline_report_rows = int(baseline_report_audit.get('rows', -1))
baseline_report_correct = int(baseline_report_audit.get('correct', -1))
baseline_report_truncated = int(baseline_report_audit.get('truncated', -1))
baseline_pred_correct = int(
    baseline_pred_audit['correct'].astype(str).str.lower().isin(['true', '1', 'yes']).sum()
)
baseline_pred_truncated = int(
    baseline_pred_audit['truncated'].astype(str).str.lower().isin(['true', '1', 'yes']).sum()
)
if (
    baseline_report_rows != len(baseline_pred_audit)
    or baseline_report_correct != baseline_pred_correct
    or baseline_report_truncated != baseline_pred_truncated
):
    raise RuntimeError(
        'Baseline report does not match predictions: '
        f'report_rows={baseline_report_rows} pred_rows={len(baseline_pred_audit)} '
        f'report_correct={baseline_report_correct} pred_correct={baseline_pred_correct} '
        f'report_truncated={baseline_report_truncated} pred_truncated={baseline_pred_truncated}'
    )
overall = baseline_per_audit[baseline_per_audit['task_type'].astype(str).eq('OVERALL')]
if len(overall) != 1:
    raise RuntimeError('Baseline per-task CSV must contain exactly one OVERALL row.')
overall_row = overall.iloc[0]
if int(overall_row['total']) != len(baseline_pred_audit) or int(overall_row['correct']) != baseline_pred_correct:
    raise RuntimeError('Baseline per-task OVERALL row does not match predictions.')

family_col = 'family' if 'family' in val.columns else 'type'
if family_col not in val.columns:
    raise RuntimeError('Validation CSV needs family or type column.')
weak = val[val[family_col].isin(WEAK_FAMILIES)].copy()
weak.to_csv(VAL_WEAK_CSV, index=False)

base_per = pd.read_csv(BASELINE_PER_TASK)
base_weak = base_per[base_per['task_type'].isin(WEAK_FAMILIES)]
BASE_WEAK_CORRECT = int(base_weak['correct'].sum())
BASE_WEAK_TOTAL = int(base_weak['total'].sum())
BASE_WEAK_TRUNCATED = int(base_weak['truncated'].sum()) if 'truncated' in base_weak.columns else 0

print('VAL rows =', len(val))
print('VAL_WEAK_CSV =', VAL_WEAK_CSV, 'exists=', VAL_WEAK_CSV.exists())
print('weak rows =', len(weak))
print('weak family counts =')
print(weak[family_col].value_counts().sort_index().to_string())
print('baseline weak correct =', BASE_WEAK_CORRECT, '/', BASE_WEAK_TOTAL)
print('baseline weak truncated =', BASE_WEAK_TRUNCATED, '/', BASE_WEAK_TOTAL)
print('=== V207B V207A ARTIFACT CHECK END ===')


In [ ]:
# CELL: download public Kaggle adapter candidates into Drive.
print('=== V207B PUBLIC KAGGLE ADAPTER DOWNLOAD START ===')
from pathlib import Path

PUBLIC_KAGGLE_MODEL_CANDIDATES = [
    # Priority 1: Huikang public adapter versions used by public competition notebooks.
    {'label': 'huikang_default_v27', 'ref': 'huikang/nemotron-adapter/Transformers/default/27', 'priority': 1},
    {'label': 'huikang_default_v26', 'ref': 'huikang/nemotron-adapter/Transformers/default/26', 'priority': 1},
    {'label': 'huikang_default_v25', 'ref': 'huikang/nemotron-adapter/Transformers/default/25', 'priority': 1},
    {'label': 'huikang_default_v24', 'ref': 'huikang/nemotron-adapter/Transformers/default/24', 'priority': 1},
    {'label': 'huikang_default_v23', 'ref': 'huikang/nemotron-adapter/Transformers/default/23', 'priority': 1},
    {'label': 'huikang_default_v22', 'ref': 'huikang/nemotron-adapter/Transformers/default/22', 'priority': 1},
    {'label': 'huikang_default_v21', 'ref': 'huikang/nemotron-adapter/Transformers/default/21', 'priority': 1},
    {'label': 'huikang_default_v20', 'ref': 'huikang/nemotron-adapter/Transformers/default/20', 'priority': 1},

    # Priority 2: Kienngx variations referenced by public notebooks and model listings.
    {'label': 'kienngx_1200samples_cot_1e_5', 'ref': 'kienngx/nemotron-nano-30b-trained/Transformers/1200samples-cot-1e-5/1', 'priority': 2},
    {'label': 'kienngx_1200samples_cot_5e_5', 'ref': 'kienngx/nemotron-nano-30b-trained/Transformers/1200samples-cot-5e-5/1', 'priority': 2},
    {'label': 'kienngx_cot_labels_3000samples', 'ref': 'kienngx/nemotron-nano-30b-trained/Transformers/cot-labels-3000samples/1', 'priority': 2},
    {'label': 'kienngx_600_samples_packing_false', 'ref': 'kienngx/nemotron-nano-30b-trained/Transformers/600-samples-packing-false/1', 'priority': 2},
    {'label': 'kienngx_1800s_lora_rank32_false', 'ref': 'kienngx/nemotron-nano-30b-trained/Transformers/1800s-lora-rank32-false/1', 'priority': 2},

    # Priority 3: extra variants. Keep default download priority at 2 to control Drive usage/time.
    {'label': 'kienngx_tinker_adapter', 'ref': 'kienngx/nemotron-nano-30b-trained/Triton/tinker-adapter/1', 'priority': 3},
    {'label': 'kienngx_2400_1e_4_lr_all_linear_packingfalse', 'ref': 'kienngx/nemotron-nano-30b-trained/Transformers/2400-1e-4_lr-all_linear-packingfalse/1', 'priority': 3},
    {'label': 'kienngx_9500s_batch1_lr1e_4', 'ref': 'kienngx/nemotron-nano-30b-trained/Transformers/9500s-batch1-lr1e-4/1', 'priority': 3},
]

PUBLIC_KAGGLE_EXPECTED_BYTES = {
    'huikang_default_v27': 1544348352,
    'huikang_default_v26': 1544348352,
    'huikang_default_v25': 1544348352,
    'huikang_default_v24': 772202848,
    'huikang_default_v23': 1544348352,
    'huikang_default_v22': 1544348352,
    'huikang_default_v21': 1544348352,
    'huikang_default_v20': 1544348352,
    'kienngx_1200samples_cot_1e_5': 3537299144,
    'kienngx_1200samples_cot_5e_5': 3537299144,
    'kienngx_cot_labels_3000samples': 3537299144,
    'kienngx_600_samples_packing_false': 1740420752,
    'kienngx_1800s_lora_rank32_false': 3479065680,
    'kienngx_tinker_adapter': 3554384888,
    'kienngx_2400_1e_4_lr_all_linear_packingfalse': 3537299144,
    'kienngx_9500s_batch1_lr1e_4': 58233016,
}
for _item in PUBLIC_KAGGLE_MODEL_CANDIDATES:
    _item['expected_bytes'] = PUBLIC_KAGGLE_EXPECTED_BYTES.get(_item['label'], 0)

def kaggle_adapter_ready(path, expected_bytes=0):
    path = Path(path)
    if not path.is_dir() or not (path / 'adapter_config.json').exists():
        return False
    model_path = path / 'adapter_model.safetensors'
    if not model_path.exists():
        model_path = path / 'adapter_model.bin'
    if not model_path.exists():
        return False
    expected_bytes = int(expected_bytes or 0)
    if expected_bytes > 0:
        min_bytes = max(1024 * 1024, int(expected_bytes * 0.98))
        if model_path.stat().st_size < min_bytes:
            print(
                'adapter_model_too_small =',
                model_path,
                'size=',
                model_path.stat().st_size,
                'min_expected=',
                min_bytes,
            )
            return False
    return True

def configure_kaggle_credentials():
    kaggle_dir = Path('/root/.kaggle')
    token_path = kaggle_dir / 'kaggle.json'
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    if token_path.exists():
        token_path.chmod(0o600)
        print('KAGGLE_CREDENTIALS_READY=True source=/root/.kaggle/kaggle.json')
        return True

    candidate_paths = [
        DRIVE_MY / 'kaggle.json',
        DRIVE_MY / 'KG1_SECRETS' / 'kaggle.json',
        DRIVE_MY / '.kaggle' / 'kaggle.json',
    ]
    env_config_dir = os.environ.get('KAGGLE_CONFIG_DIR', '').strip()
    if env_config_dir:
        candidate_paths.append(Path(env_config_dir) / 'kaggle.json')

    for src in candidate_paths:
        if src.exists():
            shutil.copy2(src, token_path)
            token_path.chmod(0o600)
            print('KAGGLE_CREDENTIALS_READY=True source=', src)
            return True

    env_username = os.environ.get('KAGGLE_USERNAME', '').strip()
    env_key = os.environ.get('KAGGLE_KEY', '').strip()
    if env_username and env_key:
        token_path.write_text(json.dumps({'username': env_username, 'key': env_key}), encoding='utf-8')
        token_path.chmod(0o600)
        os.environ.setdefault('KAGGLE_CONFIG_DIR', str(kaggle_dir))
        print('KAGGLE_CREDENTIALS_READY=True source=environment')
        return True

    try:
        from google.colab import userdata  # type: ignore
    except Exception:
        userdata = None
    if userdata is not None:
        try:
            secret_username = str(userdata.get('KAGGLE_USERNAME') or '').strip()
            secret_key = str(userdata.get('KAGGLE_KEY') or '').strip()
        except Exception:
            secret_username = ''
            secret_key = ''
        if secret_username and secret_key:
            token_path.write_text(json.dumps({'username': secret_username, 'key': secret_key}), encoding='utf-8')
            token_path.chmod(0o600)
            os.environ.setdefault('KAGGLE_CONFIG_DIR', str(kaggle_dir))
            print('KAGGLE_CREDENTIALS_READY=True source=colab_secrets')
            return True

    print('KAGGLE_CREDENTIALS_READY=False')
    print('Human action required: enable Colab Secrets KAGGLE_USERNAME/KAGGLE_KEY or place kaggle.json under MyDrive/KG1_SECRETS.')
    return False

download_status = []

if not RUN_KAGGLE_PUBLIC_DOWNLOAD:
    print('RUN_KAGGLE_PUBLIC_DOWNLOAD=False; skipping public model downloads.')
else:
    if not configure_kaggle_credentials():
        raise RuntimeError('Human action required: add Kaggle API token kaggle.json to Drive and rerun this cell.')
    if 'KAGGLE_CMD_PREFIX' not in globals():
        KAGGLE_EXE = shutil.which('kaggle')
        KAGGLE_CMD_PREFIX = [KAGGLE_EXE] if KAGGLE_EXE else [sys.executable, '-m', 'kaggle.cli']
        print('KAGGLE_CMD_PREFIX late_init =', KAGGLE_CMD_PREFIX)
    run_cmd(KAGGLE_CMD_PREFIX + ['--version'])

    selected_candidates = [
        item for item in PUBLIC_KAGGLE_MODEL_CANDIDATES
        if int(item['priority']) <= PUBLIC_DOWNLOAD_MAX_PRIORITY
    ][:PUBLIC_DOWNLOAD_MAX_CANDIDATES]
    print('selected_public_download_count =', len(selected_candidates))
    remaining_expected_bytes = sum(
        int(item.get('expected_bytes') or 0)
        for item in selected_candidates
        if not kaggle_adapter_ready(
            PUBLIC_KAGGLE_ROOT / safe_label(item['label']) / 'adapter',
            item.get('expected_bytes') or 0,
        )
    )
    usage = shutil.disk_usage(PUBLIC_KAGGLE_ROOT)
    print('download_expected_remaining_gib =', round(remaining_expected_bytes / (1024 ** 3), 2))
    print('drive_total_gib =', round(usage.total / (1024 ** 3), 2))
    print('drive_free_gib =', round(usage.free / (1024 ** 3), 2))
    if remaining_expected_bytes and usage.free < remaining_expected_bytes + 5 * 1024 ** 3:
        raise RuntimeError(
            'Human action required: not enough free Drive space for public adapter downloads. '
            f'Need about {remaining_expected_bytes / (1024 ** 3):.2f} GiB plus 5 GiB buffer; '
            f'free={usage.free / (1024 ** 3):.2f} GiB.'
        )
    for item in selected_candidates:
        label = safe_label(item['label'])
        ref = item['ref']
        target = PUBLIC_KAGGLE_ROOT / label / 'adapter'
        log = REPORT_DIR / f'download_{label}.log'
        target.mkdir(parents=True, exist_ok=True)

        already_ready = kaggle_adapter_ready(target, item.get('expected_bytes') or 0)
        print('download_start =', json.dumps({
            'label': label,
            'priority': int(item['priority']),
            'expected_gib': round(int(item.get('expected_bytes') or 0) / (1024 ** 3), 3),
            'already_ready': already_ready,
            'target': str(target),
            'log': str(log),
        }, sort_keys=True))

        if already_ready:
            status = 'already_ready'
            rc = 0
        else:
            cmd = KAGGLE_CMD_PREFIX + [
                'models',
                'instances',
                'versions',
                'download',
                ref,
                '-p',
                target,
                '--untar',
            ]
            rc = run_cmd(cmd, log_path=log, check=False)
            status = (
                'downloaded_ready'
                if rc == 0 and kaggle_adapter_ready(target, item.get('expected_bytes') or 0)
                else f'download_or_structure_failed_{rc}'
            )

        nested_ready_dirs = []
        if not kaggle_adapter_ready(target, item.get('expected_bytes') or 0):
            for dirpath, dirnames, filenames in os.walk(target):
                p = Path(dirpath)
                if kaggle_adapter_ready(p, item.get('expected_bytes') or 0):
                    nested_ready_dirs.append(str(p))

        top_files = sorted([p.name for p in target.glob('*')])[:25] if target.exists() else []
        row = {
            'label': label,
            'ref': ref,
            'priority': int(item['priority']),
            'expected_bytes': int(item.get('expected_bytes') or 0),
            'target': str(target),
            'ready': kaggle_adapter_ready(target, item.get('expected_bytes') or 0),
            'nested_ready_dirs': nested_ready_dirs,
            'status': status,
            'returncode': rc,
            'top_files': top_files,
            'log': str(log),
        }
        download_status.append(row)
        print('download_done =', json.dumps({
            'label': row['label'],
            'status': row['status'],
            'ready': row['ready'],
            'returncode': row['returncode'],
            'nested_ready_count': len(row['nested_ready_dirs']),
            'log': row['log'],
        }, sort_keys=True))

download_manifest = MANIFEST_DIR / 'v207b_public_kaggle_download_manifest.json'
download_manifest.write_text(json.dumps(download_status, indent=2, sort_keys=True), encoding='utf-8')
print('download_manifest =', download_manifest)
print('=== V207B PUBLIC KAGGLE ADAPTER DOWNLOAD END ===')


In [ ]:
# CELL: discover and register candidate adapter directories.
print('=== V207B CANDIDATE DISCOVERY START ===')
from pathlib import Path

MANUAL_CANDIDATES = [
    # Baseline sanity / tied artifacts.
    ('v194_init_duplicate', DRIVE_MY / 'KG1_NVIDIA_V202D/init_adapter_v194_rank19_build/adapter'),
    ('v194_final_keep_duplicate', DRIVE_MY / 'KG1_NVIDIA_V202D/final_v194_keep_no_submit/adapter'),
    ('v199b_candidate', DRIVE_MY / 'KG1_NVIDIA_V199B/final_adapter'),
    ('v199b_candidate_adapter', DRIVE_MY / 'KG1_NVIDIA_V199B/adapter'),

    # Common external/current adapter landing zones. Missing paths are skipped.
    ('aaitdads_my_0p86', DRIVE_MY / 'KG1_PUBLIC_ADAPTERS/aaitdads_my_0p86_adapter'),
    ('huikang_default_v27', DRIVE_MY / 'KG1_PUBLIC_ADAPTERS/huikang_default_v27/adapter'),
    ('huikang_default_v26', DRIVE_MY / 'KG1_PUBLIC_ADAPTERS/huikang_default_v26/adapter'),
    ('huikang_default_v25', DRIVE_MY / 'KG1_PUBLIC_ADAPTERS/huikang_default_v25/adapter'),
    ('huikang_default_v24', DRIVE_MY / 'KG1_PUBLIC_ADAPTERS/huikang_default_v24/adapter'),
    ('huikang_default_v23', DRIVE_MY / 'KG1_PUBLIC_ADAPTERS/huikang_default_v23/adapter'),
    ('huikang_default_v22', DRIVE_MY / 'KG1_PUBLIC_ADAPTERS/huikang_default_v22/adapter'),
    ('huikang_default_v21', DRIVE_MY / 'KG1_PUBLIC_ADAPTERS/huikang_default_v21/adapter'),
    ('huikang_default_v20', DRIVE_MY / 'KG1_PUBLIC_ADAPTERS/huikang_default_v20/adapter'),
    ('huikang_tinker_v27_legacy', DRIVE_MY / 'KG1_PUBLIC_ADAPTERS/huikang_tinker_v27/adapter'),
    ('huikang_tinker_v26_legacy', DRIVE_MY / 'KG1_PUBLIC_ADAPTERS/huikang_tinker_v26/adapter'),
    ('huikang_tinker_v20_legacy', DRIVE_MY / 'KG1_PUBLIC_ADAPTERS/huikang_tinker_v20/adapter'),
    ('kienngx_1200samples_cot_1e_5', DRIVE_MY / 'KG1_PUBLIC_ADAPTERS/kienngx_1200samples_cot_1e_5/adapter'),
    ('kienngx_1200samples_cot_5e_5', DRIVE_MY / 'KG1_PUBLIC_ADAPTERS/kienngx_1200samples_cot_5e_5/adapter'),
    ('kienngx_cot_labels_3000samples', DRIVE_MY / 'KG1_PUBLIC_ADAPTERS/kienngx_cot_labels_3000samples/adapter'),
    ('kienngx_600_samples_packing_false', DRIVE_MY / 'KG1_PUBLIC_ADAPTERS/kienngx_600_samples_packing_false/adapter'),
    ('kienngx_1800s_lora_rank32_false', DRIVE_MY / 'KG1_PUBLIC_ADAPTERS/kienngx_1800s_lora_rank32_false/adapter'),
    ('kienngx_tinker_adapter', DRIVE_MY / 'KG1_PUBLIC_ADAPTERS/kienngx_tinker_adapter/adapter'),
    ('kienngx_2400_1e_4_lr_all_linear_packingfalse', DRIVE_MY / 'KG1_PUBLIC_ADAPTERS/kienngx_2400_1e_4_lr_all_linear_packingfalse/adapter'),
    ('kienngx_9500s_batch1_lr1e_4', DRIVE_MY / 'KG1_PUBLIC_ADAPTERS/kienngx_9500s_batch1_lr1e_4/adapter'),
    ('kien_variant_legacy', DRIVE_MY / 'KG1_PUBLIC_ADAPTERS/kien_variant/adapter'),
    ('bugkeeper_v20', DRIVE_MY / 'KG1_PUBLIC_ADAPTERS/bugkeeper_v20/adapter'),
    ('dgxchen_trained', DRIVE_MY / 'KG1_PUBLIC_ADAPTERS/dgxchen_trained_adapter'),
]

DISCOVERY_ROOTS = [
    DRIVE_MY / 'KG1_PUBLIC_ADAPTERS',
    DRIVE_MY / 'KG1_NVIDIA_PUBLIC_ADAPTERS',
    DRIVE_MY / 'KG1_NVIDIA_EXTERNAL',
    DRIVE_MY / 'KG1_NVIDIA_V199B',
    DRIVE_MY / 'KG1_NVIDIA_V202D',
    DRIVE_MY / 'KG1_NVIDIA_V204',
    DRIVE_MY / 'KG1_NVIDIA_TINKER',
    DRIVE_MY / 'KG1_NVIDIA_KIEN',
]

REJECTED_SUBSTRINGS = [
    'KG1_NVIDIA_V206B',
    'KG1_NVIDIA_V206C',
    'adapter_s0p010',
    'adapter_s0p020',
    'adapter_s0p050',
    'adapter_s0p100',
]

def adapter_ready_dir(path):
    path = Path(path)
    return (
        path.is_dir()
        and (path / 'adapter_config.json').exists()
        and ((path / 'adapter_model.safetensors').exists() or (path / 'adapter_model.bin').exists())
    )

def is_rejected_path(path):
    if INCLUDE_REJECTED_V206:
        return False
    text = str(path)
    return any(chunk in text for chunk in REJECTED_SUBSTRINGS)

candidates = {}
manual_ready = 0
for label, path in MANUAL_CANDIDATES:
    path = Path(path)
    if adapter_ready_dir(path) and not is_rejected_path(path):
        manual_ready += 1
        candidates[str(path.resolve())] = {'label': safe_label(label), 'path': path}
print('manual_candidate_ready_count =', manual_ready, '/', len(MANUAL_CANDIDATES))

scan_summaries = []
for root in DISCOVERY_ROOTS:
    root_exists = root.exists()
    print('scan_root =', root, 'exists=', root_exists)
    found_before = len(candidates)
    if not root.exists():
        scan_summaries.append({'root': str(root), 'exists': False, 'scanned_dirs': 0, 'new_candidates': 0})
        continue
    scanned_dirs = 0
    for dirpath, dirnames, filenames in os.walk(root):
        scanned_dirs += 1
        if scanned_dirs > MAX_DISCOVERY_DIRS:
            print('scan_root_limit_reached', root, MAX_DISCOVERY_DIRS)
            break
        p = Path(dirpath)
        if is_rejected_path(p):
            dirnames[:] = []
            continue
        names = set(filenames)
        if 'adapter_config.json' in names and (
            'adapter_model.safetensors' in names or 'adapter_model.bin' in names
        ):
            label = safe_label(str(p.relative_to(root)))
            candidates[str(p.resolve())] = {'label': label, 'path': p}
    scan_summaries.append({
        'root': str(root),
        'exists': True,
        'scanned_dirs': scanned_dirs,
        'new_candidates': len(candidates) - found_before,
    })
    print('scan_root_done =', root, 'scanned_dirs=', scanned_dirs, 'new_candidates=', len(candidates) - found_before)

CANDIDATES = list(candidates.values())
print('candidate_count =', len(CANDIDATES))
print('candidate_preview =', json.dumps(
    [{'label': item['label'], 'path': str(item['path'])} for item in CANDIDATES[:20]],
    indent=2,
    sort_keys=True,
))
if len(CANDIDATES) > 20:
    print('candidate_preview_truncated =', len(CANDIDATES) - 20)

scan_summary_path = MANIFEST_DIR / 'v207b_discovery_scan_summary.json'
scan_summary_path.write_text(json.dumps(scan_summaries, indent=2, sort_keys=True), encoding='utf-8')
manifest_path = MANIFEST_DIR / 'v207b_discovered_candidates.json'
manifest_path.write_text(
    json.dumps(
        [{'label': x['label'], 'path': str(x['path'])} for x in CANDIDATES],
        indent=2,
        sort_keys=True,
    ),
    encoding='utf-8',
)
print('candidate_manifest =', manifest_path)
print('scan_summary =', scan_summary_path)
print('=== V207B CANDIDATE DISCOVERY END ===')


In [ ]:
# CELL: structure audit candidate adapters.
print('=== V207B STRUCTURE AUDIT START ===')
from safetensors import safe_open

VLLM_TARGET_NAMESPACE_PREFLIGHT_POLICY = 'require_safetensors_and_reject_known_mixer_targets'
print('VLLM_TARGET_NAMESPACE_PREFLIGHT_POLICY =', VLLM_TARGET_NAMESPACE_PREFLIGHT_POLICY)

UNSUPPORTED_VLLM_LORA_TARGET_PATTERNS = [
    '.mixer.gate_proj',
    '.mixer.x_proj',
    '.mixer.experts.w1',
    '.mixer.experts.w2',
    '.mixer.experts.w3',
]

UNSUPPORTED_CONFIG_TARGETS = {
    'gate_proj',
    'x_proj',
    'experts.w1',
    'experts.w2',
    'experts.w3',
}

def as_target_module_list(value):
    if value is None:
        return []
    if isinstance(value, str):
        return [value]
    if isinstance(value, (list, tuple, set)):
        return [str(item) for item in value]
    return [str(value)]

def compact_lora_weight_key(key):
    text = str(key)
    if text.startswith('base_model.model.'):
        text = text[len('base_model.model.'):]
    for suffix in ('.lora_A.weight', '.lora_B.weight', '.lora_embedding_A', '.lora_embedding_B'):
        if suffix in text:
            text = text.split(suffix, 1)[0]
    return text

def unsupported_vllm_target_examples(keys):
    examples = []
    count = 0
    for key in keys:
        compact = compact_lora_weight_key(key)
        lowered = compact.lower()
        if any(pattern in lowered for pattern in UNSUPPORTED_VLLM_LORA_TARGET_PATTERNS):
            count += 1
            if len(examples) < 8 and compact not in examples:
                examples.append(compact)
    return count, examples

def unsupported_config_target_count(target_modules):
    count = 0
    for module in target_modules:
        lowered = str(module).lower()
        if lowered in UNSUPPORTED_CONFIG_TARGETS or any(pattern.strip('.') in lowered for pattern in UNSUPPORTED_VLLM_LORA_TARGET_PATTERNS):
            count += 1
    return count

def audit_adapter(label, path):
    path = Path(path)
    cfg_path = path / 'adapter_config.json'
    model_path = path / 'adapter_model.safetensors'
    bin_path = path / 'adapter_model.bin'
    row = {
        'label': label,
        'path': str(path),
        'exists': path.exists(),
        'has_config': cfg_path.exists(),
        'has_safetensors': model_path.exists(),
        'has_bin': bin_path.exists(),
        'model_bytes': 0,
        'config_sha256': '',
        'model_sha256': '',
        'peft_type': '',
        'r': '',
        'lora_alpha': '',
        'target_modules': '',
        'base_model_name_or_path': '',
        'tensor_count': 0,
        'bad_lm_head_namespace_count': 0,
        'unsupported_config_target_count': 0,
        'unsupported_target_namespace_count': 0,
        'unsupported_target_namespace_examples': '',
        'rank_ok': False,
        'ready_for_eval': False,
        'reason': '',
    }
    if not path.exists():
        row['reason'] = 'missing_path'
        return row
    if not cfg_path.exists():
        row['reason'] = 'missing_adapter_config'
        return row
    if not model_path.exists() and not bin_path.exists():
        row['reason'] = 'missing_adapter_weights'
        return row

    cfg = json.loads(cfg_path.read_text(encoding='utf-8'))
    row['config_sha256'] = sha256_file(cfg_path, True)
    row['peft_type'] = str(cfg.get('peft_type', ''))
    row['r'] = str(cfg.get('r', ''))
    row['lora_alpha'] = str(cfg.get('lora_alpha', ''))
    target_modules = as_target_module_list(cfg.get('target_modules', ''))
    row['target_modules'] = json.dumps(target_modules, sort_keys=True)
    row['unsupported_config_target_count'] = unsupported_config_target_count(target_modules)
    row['base_model_name_or_path'] = str(cfg.get('base_model_name_or_path', ''))
    try:
        rank = int(cfg.get('r', 0))
    except Exception:
        rank = 0
    row['rank_ok'] = 0 < rank <= 32

    weight_path = model_path if model_path.exists() else bin_path
    row['model_bytes'] = int(weight_path.stat().st_size)
    row['model_sha256'] = sha256_file(weight_path, HASH_WEIGHTS)

    if not model_path.exists():
        row['reason'] = 'safetensors_required_for_namespace_preflight'
        return row

    try:
        with safe_open(str(model_path), framework='pt', device='cpu') as handle:
            keys = list(handle.keys())
        row['tensor_count'] = len(keys)
        row['bad_lm_head_namespace_count'] = sum(
            1 for key in keys if key.startswith('base_model.model.backbone.lm_head')
        )
        unsupported_count, unsupported_examples = unsupported_vllm_target_examples(keys)
        row['unsupported_target_namespace_count'] = unsupported_count
        row['unsupported_target_namespace_examples'] = json.dumps(unsupported_examples, sort_keys=True)
    except Exception as exc:
        row['reason'] = 'safetensors_open_failed:' + repr(exc)
        return row

    if not row['rank_ok']:
        row['reason'] = 'rank_missing_or_gt32'
    elif row['bad_lm_head_namespace_count']:
        row['reason'] = 'bad_lm_head_namespace'
    elif row['unsupported_config_target_count'] or row['unsupported_target_namespace_count']:
        row['reason'] = 'unsupported_vllm_lora_target_namespace'
    else:
        row['ready_for_eval'] = True
        row['reason'] = 'ready'
    return row

audit_rows = []
for item in CANDIDATES:
    audit_rows.append(audit_adapter(item['label'], item['path']))

audit_df = pd.DataFrame(audit_rows)
audit_csv = MANIFEST_DIR / 'v207b_adapter_structure_audit.csv'
audit_json = MANIFEST_DIR / 'v207b_adapter_structure_audit.json'
audit_df.to_csv(audit_csv, index=False)
audit_json.write_text(json.dumps(audit_rows, indent=2, sort_keys=True), encoding='utf-8')

print('audit_rows =', len(audit_df))
if len(audit_df):
    ready_summary = audit_df.groupby(['ready_for_eval', 'reason'], dropna=False).size().reset_index(name='count')
    print('audit_summary =')
    print(ready_summary.to_string(index=False))
    rejected_preview = audit_df[~audit_df['ready_for_eval']][['label', 'reason', 'unsupported_config_target_count', 'unsupported_target_namespace_count']].head(20)
    print('rejected_preview =')
    print(rejected_preview.to_string(index=False))
    ready_preview = audit_df[audit_df['ready_for_eval']][['label', 'r', 'tensor_count', 'model_bytes']].head(20)
    print('ready_preview =')
    print(ready_preview.to_string(index=False))
else:
    print('No candidates discovered. Add adapter paths to MANUAL_CANDIDATES or mount the Drive folder.')
print('audit_csv =', audit_csv)
print('audit_json =', audit_json)
READY_CANDIDATES = [row for row in audit_rows if row.get('ready_for_eval')]
print('ready_candidate_count =', len(READY_CANDIDATES))
print('=== V207B STRUCTURE AUDIT END ===')


In [ ]:
# CELL: weak-family screen ready candidates.
print('=== V207B WEAK FAMILY SCREEN START ===')
results = []

def weak_preflight_guard(row):
    return (
        row.get('ready_for_eval') is True
        and row.get('reason') == 'ready'
        and int(row.get('unsupported_config_target_count') or 0) == 0
        and int(row.get('unsupported_target_namespace_count') or 0) == 0
        and bool(row.get('has_safetensors'))
    )

for row in READY_CANDIDATES:
    label = safe_label(row['label'] + '_weak')
    adapter = Path(row['path'])
    out = OUT_ROOT / f'{label}_eval'
    log = REPORT_DIR / f'{label}_eval.log'
    report_json = out / f'{label}_eval_report.json'
    per_task_csv = out / f'{label}_per_task.csv'

    print('weak_eval_start =', json.dumps({
        'label': label,
        'adapter_exists': adapter.exists(),
        'out': str(out),
        'report_exists': report_json.exists(),
        'log': str(log),
    }, sort_keys=True))

    if not weak_preflight_guard(row):
        print('weak_eval_skip_preflight_guard =', json.dumps({
            'label': label,
            'reason': row.get('reason'),
            'has_safetensors': row.get('has_safetensors'),
            'unsupported_config_target_count': row.get('unsupported_config_target_count'),
            'unsupported_target_namespace_count': row.get('unsupported_target_namespace_count'),
        }, sort_keys=True))
        results.append({
            'label': label,
            'path': str(adapter),
            'status': 'skipped_preflight_guard',
            'weak_correct': 0,
            'weak_total': BASE_WEAK_TOTAL,
            'weak_delta': -BASE_WEAK_CORRECT,
            'accuracy': 0.0,
            'truncated': None,
            'truncation_rate': None,
            'promote_to_full': False,
            'report_json': str(report_json),
        })
        continue

    if FORCE_REEVAL and out.exists():
        print('FORCE_REEVAL enabled; removing previous output:', out)
        shutil.rmtree(out)

    if not report_json.exists():
        cmd = [
            sys.executable,
            ROOT / 'scripts' / 'evaluate_lora_adapter.py',
            '--solution-csv', VAL_WEAK_CSV,
            '--questions-csv', VAL_WEAK_CSV,
            '--adapter', adapter,
            '--base-model-path', MODEL_NAME,
            '--label', label,
            '--seed', '42',
            '--limit', '0',
            '--output-dir', out,
        ]
        rc = run_cmd(cmd, cwd=ROOT, log_path=log, check=False)
        if rc != 0:
            results.append({
                'label': label,
                'path': str(adapter),
                'status': f'eval_failed_{rc}',
                'weak_correct': 0,
                'weak_total': BASE_WEAK_TOTAL,
                'weak_delta': -BASE_WEAK_CORRECT,
                'accuracy': 0.0,
                'truncated': None,
                'truncation_rate': None,
                'promote_to_full': False,
                'report_json': str(report_json),
            })
            continue
    else:
        print('existing report found; skipping eval')

    report = json.loads(report_json.read_text(encoding='utf-8'))
    per = pd.read_csv(per_task_csv)
    weak_per = per[per['task_type'].isin(WEAK_FAMILIES)]
    weak_correct = int(weak_per['correct'].sum())
    weak_total = int(weak_per['total'].sum())
    weak_truncated = int(weak_per['truncated'].sum()) if 'truncated' in weak_per.columns else int(report.get('truncated', 0))
    weak_delta = weak_correct - BASE_WEAK_CORRECT
    trunc_rate = weak_truncated / weak_total if weak_total else 0.0
    promote = weak_delta > 0 and weak_total == BASE_WEAK_TOTAL

    print('weak_eval_done =', json.dumps({
        'label': label,
        'weak_correct': weak_correct,
        'weak_total': weak_total,
        'weak_delta_vs_v194': weak_delta,
        'accuracy': float(report['accuracy']),
        'weak_truncated': weak_truncated,
        'promote_to_full': bool(promote),
        'report_json': str(report_json),
    }, sort_keys=True))

    results.append({
        'label': label,
        'path': str(adapter),
        'status': 'done',
        'weak_correct': weak_correct,
        'weak_total': weak_total,
        'weak_delta': weak_delta,
        'accuracy': float(report['accuracy']),
        'truncated': weak_truncated,
        'truncation_rate': trunc_rate,
        'promote_to_full': bool(promote),
        'report_json': str(report_json),
        'predictions_csv': str(out / f'{label}_predictions.csv'),
        'per_task_csv': str(per_task_csv),
    })

weak_results = pd.DataFrame(results)
weak_csv = MANIFEST_DIR / 'v207b_weak_screen_results.csv'
weak_json = MANIFEST_DIR / 'v207b_weak_screen_results.json'
weak_results.to_csv(weak_csv, index=False)
weak_json.write_text(json.dumps(results, indent=2, sort_keys=True), encoding='utf-8')

print('WEAK SCREEN SUMMARY')
if len(weak_results):
    print(weak_results[['label', 'status', 'weak_correct', 'weak_total', 'weak_delta', 'accuracy', 'truncated', 'promote_to_full']].sort_values(['promote_to_full', 'weak_delta'], ascending=[False, False]).head(20).to_string(index=False))
else:
    print('No ready candidates to evaluate.')
print('weak_csv =', weak_csv)
print('weak_json =', weak_json)
FULL_CANDIDATES = [item for item in results if item.get('promote_to_full')]
print('full_candidate_count =', len(FULL_CANDIDATES))
print('=== V207B WEAK FAMILY SCREEN END ===')


In [ ]:
# CELL: full 947-row gate only for weak-positive candidates.
print('=== V207B FULL GATE START ===')
full_results = []

if not RUN_FULL_FOR_POSITIVE:
    print('RUN_FULL_FOR_POSITIVE=False; skipping full gate by configuration.')
elif not FULL_CANDIDATES:
    print('No weak-positive candidates. Full 947-row evaluation skipped.')
else:
    for item in FULL_CANDIDATES:
        base_label = safe_label(item['label'].replace('_weak', ''))
        adapter = Path(item['path'])
        eval_label = safe_label(base_label + '_full')
        eval_out = OUT_ROOT / f'{eval_label}_eval'
        eval_log = REPORT_DIR / f'{eval_label}_eval.log'
        report_json = eval_out / f'{eval_label}_eval_report.json'
        candidate_predictions = eval_out / f'{eval_label}_predictions.csv'

        print('full_eval_start =', json.dumps({
            'label': eval_label,
            'adapter': str(adapter),
            'eval_out': str(eval_out),
            'eval_log': str(eval_log),
        }, sort_keys=True))

        if FORCE_REEVAL and eval_out.exists():
            print('FORCE_REEVAL enabled; removing previous full output:', eval_out)
            shutil.rmtree(eval_out)

        if not report_json.exists():
            cmd = [
                sys.executable,
                ROOT / 'scripts' / 'evaluate_lora_adapter.py',
                '--solution-csv', VAL_CSV,
                '--questions-csv', VAL_CSV,
                '--adapter', adapter,
                '--base-model-path', MODEL_NAME,
                '--label', eval_label,
                '--seed', '42',
                '--limit', '0',
                '--output-dir', eval_out,
            ]
            rc = run_cmd(cmd, cwd=ROOT, log_path=eval_log, check=False)
            if rc != 0:
                full_results.append({'label': eval_label, 'status': f'eval_failed_{rc}', 'path': str(adapter)})
                continue
        else:
            print('existing full report found; skipping eval')

        gate_out = OUT_ROOT / f'{eval_label}_gate'
        gate_log = REPORT_DIR / f'{eval_label}_gate.log'
        cmd = [
            sys.executable,
            ROOT / 'scripts' / 'solve_rate_gate.py',
            '--solution-csv', VAL_CSV,
            '--baseline-predictions', BASELINE_PREDICTIONS,
            '--candidate-predictions', candidate_predictions,
            '--family-regression-tolerance', '0.0',
            '--min-net-gain', '0.0',
            '--min-boxed-rate', '0.98',
            '--output-dir', gate_out,
        ]
        rc = run_cmd(cmd, cwd=ROOT, log_path=gate_log, check=False)
        gate_report = gate_out / 'solve_rate_gate_report.json'
        gate_payload = json.loads(gate_report.read_text(encoding='utf-8')) if gate_report.exists() else {}
        full_results.append({
            'label': eval_label,
            'status': 'gate_approve' if rc == 0 else f'gate_reject_{rc}',
            'path': str(adapter),
            'gate_report': str(gate_report),
            'approved': bool(gate_payload.get('approved', False)),
            'net_gain': gate_payload.get('comparison', {}).get('net_gain'),
            'reasons': gate_payload.get('reasons', []),
        })
        print('full_gate_done =', json.dumps(full_results[-1], sort_keys=True))

full_df = pd.DataFrame(full_results)
full_csv = MANIFEST_DIR / 'v207b_full_gate_results.csv'
full_json = MANIFEST_DIR / 'v207b_full_gate_results.json'
full_df.to_csv(full_csv, index=False)
full_json.write_text(json.dumps(full_results, indent=2, sort_keys=True), encoding='utf-8')

print('FULL GATE SUMMARY')
if len(full_df):
    print(full_df[['label', 'status', 'approved', 'net_gain', 'gate_report']].to_string(index=False))
else:
    print('No full-gate candidates were evaluated.')
print('full_csv =', full_csv)
print('full_json =', full_json)
print('=== V207B FULL GATE END ===')


In [ ]:
# CELL: final V207B summary. This cell does not submit.
print('=== V207B FINAL SUMMARY START ===')
summary = {
    'version': VERSION,
    'status': 'v207b_external_adapter_triage_completed',
    'submit_disabled': True,
    'v207a_root': str(V207A_ROOT),
    'out_root': str(OUT_ROOT),
    'report_dir': str(REPORT_DIR),
    'manifest_dir': str(MANIFEST_DIR),
    'log_policy': LOG_POLICY,
    'candidate_manifest': str(MANIFEST_DIR / 'v207b_discovered_candidates.json'),
    'structure_audit_csv': str(MANIFEST_DIR / 'v207b_adapter_structure_audit.csv'),
    'weak_screen_csv': str(MANIFEST_DIR / 'v207b_weak_screen_results.csv'),
    'full_gate_csv': str(MANIFEST_DIR / 'v207b_full_gate_results.csv'),
    'full_candidate_count': len(FULL_CANDIDATES) if 'FULL_CANDIDATES' in globals() else 0,
    'next_human_action': (
        'Review any approved full gate candidate and explicitly approve Kaggle submission.'
        if 'full_results' in globals() and any(x.get('approved') for x in full_results)
        else 'No submission candidate approved. Add more external adapters or stop.'
    ),
}
summary_path = OUT_ROOT / 'V207B_FINAL_RUN_SUMMARY.json'
summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True), encoding='utf-8')
print(json.dumps(summary, indent=2, sort_keys=True))
print('summary_path =', summary_path)
print('=== V207B FINAL SUMMARY END ===')
